In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.0 MB/s eta 0:00:00


In [ ]:
"""
LSTM Model Exploration for Sri Lankan Tourism Arrivals Prediction
==================================================================
This module implements a production-ready LSTM model with hyperparameter tuning,
time-series aware validation, and comprehensive evaluation metrics.

Author: ML Research Team
Date: December 2025
"""

import numpy as np
import pandas as pd
import logging
import json
import warnings
from datetime import datetime
from typing import Tuple, Dict, List, Any
import os

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


class LSTMModelExplorer:
    """
    LSTM Model Explorer for time-series forecasting with comprehensive
    preprocessing, hyperparameter tuning, and evaluation.
    """

    def __init__(self, data_path: str, output_dir: str = 'lstm_output'):
        """
        Initialize the LSTM Model Explorer.

        Args:
            data_path: Path to preprocessed dataset CSV
            output_dir: Directory for saving outputs
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.setup_logging()
        self.setup_output_directory()

        # Model artifacts
        self.scaler_X = None
        self.scaler_y = None
        self.best_model = None
        self.best_params = None
        self.history = None

        # Data splits
        self.X_train, self.X_val, self.X_test = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.train_dates, self.val_dates, self.test_dates = None, None, None

        self.logger.info("LSTM Model Explorer initialized successfully")

    def setup_logging(self):
        """Configure logging with both file and console handlers."""
        log_format = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'

        # Create logger
        self.logger = logging.getLogger('LSTMModelExplorer')
        self.logger.setLevel(logging.INFO)

        # Console handler
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter(log_format))

        # File handler
        file_handler = logging.FileHandler(f'lstm_exploration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(logging.Formatter(log_format))

        self.logger.addHandler(console_handler)
        self.logger.addHandler(file_handler)

    def setup_output_directory(self):
        """Create output directory structure."""
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'metrics'), exist_ok=True)
        self.logger.info(f"Output directory created: {self.output_dir}")

    def load_data(self) -> pd.DataFrame:
        """Load and validate preprocessed dataset."""
        self.logger.info(f"Loading data from {self.data_path}")

        try:
            df = pd.read_csv(self.data_path)
            self.logger.info(f"Data loaded successfully. Shape: {df.shape}")
            self.logger.info(f"Columns: {list(df.columns)}")
            self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

            # Convert date column to datetime
            df['date'] = pd.to_datetime(df['date'])

            # Sort by date to ensure chronological order
            df = df.sort_values('date').reset_index(drop=True)

            return df

        except Exception as e:
            self.logger.error(f"Error loading data: {str(e)}")
            raise

    def create_lstm_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create LSTM-specific features including lag features and rolling statistics.

        Args:
            df: Input dataframe

        Returns:
            DataFrame with LSTM-specific features
        """
        self.logger.info("Creating LSTM-specific features")

        df = df.copy()

        # Temporal features from date
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week
        df['year'] = df['date'].dt.year

        # Cyclical encoding for temporal features
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

        # Lag features for arrivals (using shifted values)
        for lag in [1, 7, 14, 30]:
            df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

        # Rolling statistics
        for window in [7, 14, 30]:
            df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(window=window, min_periods=1).mean()
            df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(window=window, min_periods=1).std()

        # Fill NaN values from lag features with forward fill then backward fill
        df = df.fillna(method='ffill').fillna(method='bfill')

        self.logger.info(f"LSTM features created. New shape: {df.shape}")

        return df

    def prepare_sequences(self, data: np.ndarray, sequence_length: int) -> np.ndarray:
        """
        Prepare sequences for LSTM input.

        Args:
            data: Input data array (n_samples, n_features)
            sequence_length: Length of sequences (lookback window)

        Returns:
            Array of sequences (n_sequences, sequence_length, n_features)
        """
        sequences = []
        for i in range(len(data) - sequence_length + 1):
            sequences.append(data[i:i + sequence_length])

        return np.array(sequences)

    def timeseries_train_test_split(self, df: pd.DataFrame,
                                   train_ratio: float = 0.7,
                                   val_ratio: float = 0.15,
                                   test_ratio: float = 0.15,
                                   sequence_length: int = 30) -> Tuple:
        """
        Perform time-series aware train/validation/test split with sequence creation.

        Args:
            df: Input dataframe
            train_ratio: Proportion for training set
            val_ratio: Proportion for validation set
            test_ratio: Proportion for test set
            sequence_length: Lookback window for LSTM

        Returns:
            Tuple of train, validation, and test sets
        """
        self.logger.info(f"Performing time-series split: {train_ratio}/{val_ratio}/{test_ratio}")
        self.logger.info(f"Sequence length (lookback): {sequence_length}")

        # Verify ratios sum to 1
        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

        n = len(df)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        # Split data
        train_df = df.iloc[:train_end].copy()
        val_df = df.iloc[train_end:val_end].copy()
        test_df = df.iloc[val_end:].copy()

        self.logger.info(f"Train set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})")
        self.logger.info(f"Validation set: {len(val_df)} samples ({val_df['date'].min()} to {val_df['date'].max()})")
        self.logger.info(f"Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})")

        # Store dates
        self.train_dates = train_df['date'].values
        self.val_dates = val_df['date'].values
        self.test_dates = test_df['date'].values

        # Define feature columns (exclude date and target)
        feature_cols = [col for col in df.columns if col not in ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']]

        # Extract features and target
        X_train = train_df[feature_cols].values
        X_val = val_df[feature_cols].values
        X_test = test_df[feature_cols].values

        y_train = train_df['arrivals'].values
        y_val = val_df['arrivals'].values
        y_test = test_df['arrivals'].values

        # Scale features
        self.logger.info("Scaling features using StandardScaler")
        self.scaler_X = StandardScaler()
        X_train_scaled = self.scaler_X.fit_transform(X_train)
        X_val_scaled = self.scaler_X.transform(X_val)
        X_test_scaled = self.scaler_X.transform(X_test)

        # Scale target
        self.logger.info("Scaling target using MinMaxScaler")
        self.scaler_y = MinMaxScaler()
        y_train_scaled = self.scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val_scaled = self.scaler_y.transform(y_val.reshape(-1, 1)).flatten()
        y_test_scaled = self.scaler_y.transform(y_test.reshape(-1, 1)).flatten()

        # Create sequences for LSTM
        self.logger.info("Creating sequences for LSTM")
        X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
        X_val_seq = self.prepare_sequences(X_val_scaled, sequence_length)
        X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)

        # Align y with sequences (remove first sequence_length-1 samples)
        y_train_seq = y_train_scaled[sequence_length - 1:]
        y_val_seq = y_val_scaled[sequence_length - 1:]
        y_test_seq = y_test_scaled[sequence_length - 1:]

        self.logger.info(f"Final shapes - X_train: {X_train_seq.shape}, y_train: {y_train_seq.shape}")
        self.logger.info(f"Final shapes - X_val: {X_val_seq.shape}, y_val: {y_val_seq.shape}")
        self.logger.info(f"Final shapes - X_test: {X_test_seq.shape}, y_test: {y_test_seq.shape}")

        return (X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq)

    def build_lstm_model(self, input_shape: Tuple[int, int],
                        lstm_units: List[int] = [128, 64],
                        dropout_rate: float = 0.2,
                        learning_rate: float = 0.001) -> keras.Model:
        """
        Build LSTM model architecture.

        Args:
            input_shape: Shape of input (sequence_length, n_features)
            lstm_units: List of units for each LSTM layer
            dropout_rate: Dropout rate for regularization
            learning_rate: Learning rate for optimizer

        Returns:
            Compiled Keras model
        """
        model = Sequential(name='LSTM_Tourism_Predictor')

        # First LSTM layer
        model.add(LSTM(units=lstm_units[0],
                      return_sequences=len(lstm_units) > 1,
                      input_shape=input_shape,
                      name='lstm_1'))
        model.add(BatchNormalization(name='batch_norm_1'))
        model.add(Dropout(dropout_rate, name='dropout_1'))

        # Additional LSTM layers
        for i, units in enumerate(lstm_units[1:], start=2):
            return_seq = i < len(lstm_units)
            model.add(LSTM(units=units,
                          return_sequences=return_seq,
                          name=f'lstm_{i}'))
            model.add(BatchNormalization(name=f'batch_norm_{i}'))
            model.add(Dropout(dropout_rate, name=f'dropout_{i}'))

        # Dense layers
        model.add(Dense(32, activation='relu', name='dense_1'))
        model.add(Dropout(dropout_rate, name='dropout_final'))
        model.add(Dense(1, activation='linear', name='output'))

        # Compile model
        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

        return model

    def hyperparameter_tuning(self, X_train: np.ndarray, y_train: np.ndarray,
                            X_val: np.ndarray, y_val: np.ndarray) -> Dict[str, Any]:
        """
        Perform hyperparameter tuning using grid search.

        Args:
            X_train: Training features
            y_train: Training target
            X_val: Validation features
            y_val: Validation target

        Returns:
            Dictionary of best hyperparameters
        """
        self.logger.info("Starting hyperparameter tuning")

        # Define hyperparameter search space
        param_grid = {
            'lstm_units': [[128, 64], [256, 128], [128, 64, 32]],
            'dropout_rate': [0.2, 0.3],
            'learning_rate': [0.001, 0.0005],
            'batch_size': [32, 64]
        }

        best_val_loss = float('inf')
        best_params = None

        total_combinations = (len(param_grid['lstm_units']) *
                            len(param_grid['dropout_rate']) *
                            len(param_grid['learning_rate']) *
                            len(param_grid['batch_size']))

        self.logger.info(f"Testing {total_combinations} hyperparameter combinations")

        trial = 0
        for lstm_units in param_grid['lstm_units']:
            for dropout_rate in param_grid['dropout_rate']:
                for learning_rate in param_grid['learning_rate']:
                    for batch_size in param_grid['batch_size']:
                        trial += 1

                        self.logger.info(f"\nTrial {trial}/{total_combinations}")
                        self.logger.info(f"Params: lstm_units={lstm_units}, dropout={dropout_rate}, "
                                       f"lr={learning_rate}, batch_size={batch_size}")

                        # Build model
                        model = self.build_lstm_model(
                            input_shape=(X_train.shape[1], X_train.shape[2]),
                            lstm_units=lstm_units,
                            dropout_rate=dropout_rate,
                            learning_rate=learning_rate
                        )

                        # Callbacks
                        early_stop = EarlyStopping(monitor='val_loss', patience=10,
                                                  restore_best_weights=True, verbose=0)
                        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                     patience=5, min_lr=1e-7, verbose=0)

                        # Train
                        history = model.fit(
                            X_train, y_train,
                            validation_data=(X_val, y_val),
                            epochs=100,
                            batch_size=batch_size,
                            callbacks=[early_stop, reduce_lr],
                            verbose=0
                        )

                        # Evaluate
                        val_loss = min(history.history['val_loss'])
                        self.logger.info(f"Validation loss: {val_loss:.6f}")

                        # Update best parameters
                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            best_params = {
                                'lstm_units': lstm_units,
                                'dropout_rate': dropout_rate,
                                'learning_rate': learning_rate,
                                'batch_size': batch_size,
                                'val_loss': val_loss
                            }
                            self.logger.info(f"*** New best parameters found! Val loss: {val_loss:.6f} ***")

        self.logger.info(f"\nHyperparameter tuning completed")
        self.logger.info(f"Best parameters: {best_params}")
        self.best_params = best_params

        # Save best parameters
        with open(os.path.join(self.output_dir, 'best_hyperparameters.json'), 'w') as f:
            json.dump(best_params, f, indent=4)

        return best_params

    def train_final_model(self, X_train: np.ndarray, y_train: np.ndarray,
                         X_val: np.ndarray, y_val: np.ndarray,
                         params: Dict[str, Any]) -> keras.Model:
        """
        Train final model with best hyperparameters.

        Args:
            X_train: Training features
            y_train: Training target
            X_val: Validation features
            y_val: Validation target
            params: Best hyperparameters

        Returns:
            Trained Keras model
        """
        self.logger.info("Training final model with best hyperparameters")

        # Build model
        model = self.build_lstm_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            lstm_units=params['lstm_units'],
            dropout_rate=params['dropout_rate'],
            learning_rate=params['learning_rate']
        )

        # Print model summary
        self.logger.info("\nModel Architecture:")
        model.summary(print_fn=lambda x: self.logger.info(x))

        # Callbacks
        model_path = os.path.join(self.output_dir, 'models', 'best_lstm_model.h5')
        checkpoint = ModelCheckpoint(model_path, monitor='val_loss',
                                    save_best_only=True, verbose=1)
        early_stop = EarlyStopping(monitor='val_loss', patience=20,
                                  restore_best_weights=True, verbose=1)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=7, min_lr=1e-7, verbose=1)

        # Train
        self.logger.info("Starting training...")
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=200,
            batch_size=params['batch_size'],
            callbacks=[checkpoint, early_stop, reduce_lr],
            verbose=1
        )

        self.history = history.history
        self.best_model = model

        self.logger.info("Training completed")

        return model

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray,
                         set_name: str = '') -> Dict[str, float]:
        """
        Calculate evaluation metrics.

        Args:
            y_true: True values
            y_pred: Predicted values
            set_name: Name of dataset (train/val/test)

        Returns:
            Dictionary of metrics
        """
        # Inverse transform predictions and true values
        y_true_original = self.scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
        y_pred_original = self.scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

        # Calculate metrics
        mse = mean_squared_error(y_true_original, y_pred_original)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true_original, y_pred_original)
        mape = mean_absolute_percentage_error(y_true_original, y_pred_original) * 100

        metrics = {
            'MSE': float(mse),
            'RMSE': float(rmse),
            'R2': float(r2),
            'MAPE': float(mape)
        }

        self.logger.info(f"\n{set_name} Metrics:")
        for metric, value in metrics.items():
            self.logger.info(f"  {metric}: {value:.4f}")

        return metrics

    def time_series_cross_validation(self, df: pd.DataFrame,
                                     n_splits: int = 5,
                                     sequence_length: int = 30) -> List[Dict[str, float]]:
        """
        Perform time-series cross-validation.

        Args:
            df: Input dataframe
            n_splits: Number of CV splits
            sequence_length: Lookback window

        Returns:
            List of metrics for each fold
        """
        self.logger.info(f"\nPerforming time-series cross-validation with {n_splits} splits")

        feature_cols = [col for col in df.columns if col not in ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']]

        n = len(df)
        fold_size = n // (n_splits + 1)

        cv_metrics = []

        for fold in range(n_splits):
            self.logger.info(f"\n--- Fold {fold + 1}/{n_splits} ---")

            # Define train/test split for this fold
            test_start = fold_size * (fold + 1)
            test_end = test_start + fold_size

            train_df = df.iloc[:test_start].copy()
            test_df = df.iloc[test_start:test_end].copy()

            if len(train_df) < sequence_length or len(test_df) < sequence_length:
                self.logger.warning(f"Fold {fold + 1} skipped due to insufficient data")
                continue

            # Prepare data
            X_train = train_df[feature_cols].values
            X_test = test_df[feature_cols].values
            y_train = train_df['arrivals'].values
            y_test = test_df['arrivals'].values

            # Scale
            scaler_X = StandardScaler()
            scaler_y = MinMaxScaler()

            X_train_scaled = scaler_X.fit_transform(X_train)
            X_test_scaled = scaler_X.transform(X_test)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

            # Create sequences
            X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
            X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)
            y_train_seq = y_train_scaled[sequence_length - 1:]
            y_test_seq = y_test_scaled[sequence_length - 1:]

            # Build and train model (using best params if available)
            if self.best_params:
                params = self.best_params
            else:
                params = {
                    'lstm_units': [128, 64],
                    'dropout_rate': 0.2,
                    'learning_rate': 0.001,
                    'batch_size': 32
                }

            model = self.build_lstm_model(
                input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
                lstm_units=params['lstm_units'],
                dropout_rate=params['dropout_rate'],
                learning_rate=params['learning_rate']
            )

            early_stop = EarlyStopping(monitor='loss', patience=10,
                                      restore_best_weights=True, verbose=0)

            model.fit(X_train_seq, y_train_seq,
                     epochs=50,
                     batch_size=params['batch_size'],
                     callbacks=[early_stop],
                     verbose=0)

            # Predict and evaluate
            y_pred = model.predict(X_test_seq, verbose=0).flatten()

            # Calculate metrics
            y_true_original = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
            y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

            fold_metrics = {
                'fold': fold + 1,
                'MSE': float(mean_squared_error(y_true_original, y_pred_original)),
                'RMSE': float(np.sqrt(mean_squared_error(y_true_original, y_pred_original))),
                'R2': float(r2_score(y_true_original, y_pred_original)),
                'MAPE': float(mean_absolute_percentage_error(y_true_original, y_pred_original) * 100)
            }

            cv_metrics.append(fold_metrics)

            self.logger.info(f"Fold {fold + 1} metrics:")
            for metric, value in fold_metrics.items():
                if metric != 'fold':
                    self.logger.info(f"  {metric}: {value:.4f}")

        # Calculate average metrics
        if cv_metrics:
            avg_metrics = {
                'MSE': np.mean([m['MSE'] for m in cv_metrics]),
                'RMSE': np.mean([m['RMSE'] for m in cv_metrics]),
                'R2': np.mean([m['R2'] for m in cv_metrics]),
                'MAPE': np.mean([m['MAPE'] for m in cv_metrics])
            }

            self.logger.info(f"\nAverage CV Metrics:")
            for metric, value in avg_metrics.items():
                self.logger.info(f"  {metric}: {value:.4f}")

        return cv_metrics

    def run_full_pipeline(self, sequence_length: int = 30,
                         perform_cv: bool = True):
        """
        Execute the complete LSTM model exploration pipeline.

        Args:
            sequence_length: Lookback window for LSTM
            perform_cv: Whether to perform cross-validation
        """
        self.logger.info("="*80)
        self.logger.info("STARTING LSTM MODEL EXPLORATION PIPELINE")
        self.logger.info("="*80)

        # 1. Load Data
        self.logger.info("\n[STEP 1] Loading Data")
        df = self.load_data()

        # 2. Create LSTM Features
        self.logger.info("\n[STEP 2] Creating LSTM-Specific Features")
        df = self.create_lstm_features(df)

        # 3. Train/Val/Test Split
        self.logger.info("\n[STEP 3] Splitting Data (70/15/15)")
        X_train, y_train, X_val, y_val, X_test, y_test = self.timeseries_train_test_split(
            df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
            sequence_length=sequence_length
        )

        # Store splits
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test

        # 4. Hyperparameter Tuning
        self.logger.info("\n[STEP 4] Hyperparameter Tuning")
        best_params = self.hyperparameter_tuning(X_train, y_train, X_val, y_val)

        # 5. Train Final Model
        self.logger.info("\n[STEP 5] Training Final Model")
        model = self.train_final_model(X_train, y_train, X_val, y_val, best_params)

        # 6. Evaluation
        self.logger.info("\n[STEP 6] Model Evaluation")

        # Predictions
        y_train_pred = model.predict(X_train, verbose=0).flatten()
        y_val_pred = model.predict(X_val, verbose=0).flatten()
        y_test_pred = model.predict(X_test, verbose=0).flatten()

        # Calculate metrics
        train_metrics = self.calculate_metrics(y_train, y_train_pred, 'Training Set')
        val_metrics = self.calculate_metrics(y_val, y_val_pred, 'Validation Set')
        test_metrics = self.calculate_metrics(y_test, y_test_pred, 'Test Set')

        # 7. Cross-Validation
        cv_metrics = None
        if perform_cv:
            self.logger.info("\n[STEP 7] Time-Series Cross-Validation")
            cv_metrics = self.time_series_cross_validation(df, n_splits=5,
                                                          sequence_length=sequence_length)

        # 8. Save Results
        self.logger.info("\n[STEP 8] Saving Results")

        all_metrics = {
            'train_metrics': train_metrics,
            'validation_metrics': val_metrics,
            'test_metrics': test_metrics,
            'cv_metrics': cv_metrics if cv_metrics else [],
            'best_hyperparameters': best_params,
            'training_history': {
                'loss': [float(x) for x in self.history['loss']],
                'val_loss': [float(x) for x in self.history['val_loss']]
            }
        }

        metrics_path = os.path.join(self.output_dir, 'metrics', 'all_metrics.json')
        with open(metrics_path, 'w') as f:
            json.dump(all_metrics, f, indent=4)

        self.logger.info(f"All metrics saved to {metrics_path}")

        # Save predictions
        predictions_df = pd.DataFrame({
            'y_true': self.scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten(),
            'y_pred': self.scaler_y.inverse_transform(y_test_pred.reshape(-1, 1)).flatten()
        })
        pred_path = os.path.join(self.output_dir, 'test_predictions.csv')
        predictions_df.to_csv(pred_path, index=False)
        self.logger.info(f"Test predictions saved to {pred_path}")

        self.logger.info("\n" + "="*80)
        self.logger.info("LSTM MODEL EXPLORATION PIPELINE COMPLETED SUCCESSFULLY")
        self.logger.info("="*80)

        return all_metrics


def main():
    """Main execution function."""
    # Configuration
    DATA_PATH = 'preprocessed-dataset.csv'
    OUTPUT_DIR = 'lstm_output'
    SEQUENCE_LENGTH = 30  # 30-day lookback window
    PERFORM_CV = True

    # Initialize explorer
    explorer = LSTMModelExplorer(data_path=DATA_PATH, output_dir=OUTPUT_DIR)

    # Run pipeline
    metrics = explorer.run_full_pipeline(
        sequence_length=SEQUENCE_LENGTH,
        perform_cv=PERFORM_CV
    )

    print("\n" + "="*80)
    print("EXECUTION COMPLETED")
    print("="*80)
    print(f"\nFinal Test Set Metrics:")
    for metric, value in metrics['test_metrics'].items():
        print(f"  {metric}: {value:.4f}")


if __name__ == '__main__':
    main()

2026-03-18 04:14:26,966 - LSTMModelExplorer - INFO - Output directory created: lstm_output
INFO:LSTMModelExplorer:Output directory created: lstm_output
2026-03-18 04:14:26,972 - LSTMModelExplorer - INFO - LSTM Model Explorer initialized successfully
INFO:LSTMModelExplorer:LSTM Model Explorer initialized successfully
2026-03-18 04:14:26,975 - LSTMModelExplorer - INFO - ================================================================================
INFO:LSTMModelExplorer:================================================================================
2026-03-18 04:14:26,977 - LSTMModelExplorer - INFO - STARTING LSTM MODEL EXPLORATION PIPELINE
INFO:LSTMModelExplorer:STARTING LSTM MODEL EXPLORATION PIPELINE
2026-03-18 04:14:26,980 - LSTMModelExplorer - INFO - ================================================================================
INFO:LSTMModelExplorer:================================================================================
2026-03-18 04:14:26,984 - LSTMModelExplorer - IN

2026-03-18 04:30:07,928 - LSTMModelExplorer - INFO - Model: "LSTM_Tourism_Predictor"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 30, 128)        │        84,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 30, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 30, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼────────

Epoch 1/200
59/63 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1.2671 - mae: 0.8594
Epoch 1: val_loss improved from None to 0.02215, saving model to lstm_output/models/best_lstm_model.h5



Epoch 1: finished saving model to lstm_output/models/best_lstm_model.h5
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - loss: 0.7152 - mae: 0.6405 - val_loss: 0.0222 - val_mae: 0.1247 - learning_rate: 0.0010
Epoch 2/200
62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.2926 - mae: 0.4255
Epoch 2: val_loss improved from 0.02215 to 0.01890, saving model to lstm_output/models/best_lstm_model.h5



Epoch 2: finished saving model to lstm_output/models/best_lstm_model.h5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.2576 - mae: 0.3979 - val_loss: 0.0189 - val_mae: 0.1150 - learning_rate: 0.0010
Epoch 3/200
59/63 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1794 - mae: 0.3350
Epoch 3: val_loss improved from 0.01890 to 0.01456, saving model to lstm_output/models/best_lstm_model.h5



Epoch 3: finished saving model to lstm_output/models/best_lstm_model.h5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.1677 - mae: 0.3232 - val_loss: 0.0146 - val_mae: 0.1024 - learning_rate: 0.0010
Epoch 4/200
59/63 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.1232 - mae: 0.2794
Epoch 4: val_loss did not improve from 0.01456
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.1136 - mae: 0.2671 - val_loss: 0.0195 - val_mae: 0.1229 - learning_rate: 0.0010
Epoch 5/200
61/63 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0892 - mae: 0.2345
Epoch 5: val_loss did not improve from 0.01456
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0861 - mae: 0.2303 - val_loss: 0.0238 - val_mae: 0.1322 - learning_rate: 0.0010
Epoch 6/200
62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0724 - mae: 0.2097
Epoch 6: val_loss did not improve from 0.01456
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0713 - mae: 0.2090 - val_loss: 0.0263 - val_mae: 0.1327 - learning_rate: 0.0010
Epoch 7/200
59/63 ━━━━━


Epoch 14: finished saving model to lstm_output/models/best_lstm_model.h5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0258 - mae: 0.1243 - val_loss: 0.0135 - val_mae: 0.0867 - learning_rate: 5.0000e-04
Epoch 15/200
61/63 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0248 - mae: 0.1217
Epoch 15: val_loss did not improve from 0.01346
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0260 - mae: 0.1247 - val_loss: 0.0148 - val_mae: 0.0917 - learning_rate: 5.0000e-04
Epoch 16/200
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0222 - mae: 0.1147
Epoch 16: val_loss did not improve from 0.01346
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0225 - mae: 0.1160 - val_loss: 0.0138 - val_mae: 0.0883 - learning_rate: 5.0000e-04
Epoch 17/200
62/63 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0222 - mae: 0.1139
Epoch 17: val_loss did not improve from 0.01346
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0219 - mae: 0.1139 - val_loss: 0.0144 - val_mae: 0.0894 - learning_rate: 5.0000e-04


2026-03-18 04:30:48,289 - LSTMModelExplorer - INFO - Training completed
INFO:LSTMModelExplorer:Training completed
2026-03-18 04:30:48,292 - LSTMModelExplorer - INFO - 
[STEP 6] Model Evaluation
INFO:LSTMModelExplorer:
[STEP 6] Model Evaluation
2026-03-18 04:30:49,667 - LSTMModelExplorer - INFO - 
Training Set Metrics:
INFO:LSTMModelExplorer:
Training Set Metrics:
2026-03-18 04:30:49,670 - LSTMModelExplorer - INFO -   MSE: 1185639.5611
INFO:LSTMModelExplorer:  MSE: 1185639.5611
2026-03-18 04:30:49,673 - LSTMModelExplorer - INFO -   RMSE: 1088.8708
INFO:LSTMModelExplorer:  RMSE: 1088.8708
2026-03-18 04:30:49,676 - LSTMModelExplorer - INFO -   R2: 0.7809
INFO:LSTMModelExplorer:  R2: 0.7809
2026-03-18 04:30:49,679 - LSTMModelExplorer - INFO -   MAPE: 20369445583066882048.0000
INFO:LSTMModelExplorer:  MAPE: 20369445583066882048.0000
2026-03-18 04:30:49,686 - LSTMModelExplorer - INFO - 
Validation Set Metrics:
INFO:LSTMModelExplorer:
Validation Set Metrics:
2026-03-18 04:30:49,688 - LSTMMode


EXECUTION COMPLETED

Final Test Set Metrics:
  MSE: 4938436.2137
  RMSE: 2222.2593
  R2: -0.1261
  MAPE: 29.1002


In [ ]:
"""
BiLSTM Model Exploration for Sri Lankan Tourism Arrivals Prediction
====================================================================
This module implements a production-ready Bidirectional LSTM model with hyperparameter tuning,
time-series aware validation, and comprehensive evaluation metrics.

Bidirectional LSTMs process sequences in both forward and backward directions,
capturing dependencies from both past and future context.

Author: ML Research Team
Date: December 2025
"""

import numpy as np
import pandas as pd
import logging
import json
import warnings
from datetime import datetime
from typing import Tuple, Dict, List, Any
import os

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# Suppress warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


class BiLSTMModelExplorer:
    """
    Bidirectional LSTM Model Explorer for time-series forecasting with comprehensive
    preprocessing, hyperparameter tuning, and evaluation.

    BiLSTM processes sequences in both forward and backward directions, allowing
    the model to capture dependencies from both past and future time steps.
    """

    def __init__(self, data_path: str, output_dir: str = 'bilstm_output'):
        """
        Initialize the BiLSTM Model Explorer.

        Args:
            data_path: Path to preprocessed dataset CSV
            output_dir: Directory for saving outputs
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.setup_logging()
        self.setup_output_directory()

        # Model artifacts
        self.scaler_X = None
        self.scaler_y = None
        self.best_model = None
        self.best_params = None
        self.history = None

        # Data splits
        self.X_train, self.X_val, self.X_test = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.train_dates, self.val_dates, self.test_dates = None, None, None

        self.logger.info("BiLSTM Model Explorer initialized successfully")

    def setup_logging(self):
        """Configure logging with both file and console handlers."""
        log_format = '%(asctime)s - %(name)s - %(levelname)s - %(message)s'

        # Create logger
        self.logger = logging.getLogger('BiLSTMModelExplorer')
        self.logger.setLevel(logging.INFO)

        # Console handler
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter(log_format))

        # File handler
        file_handler = logging.FileHandler(f'bilstm_exploration_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(logging.Formatter(log_format))

        self.logger.addHandler(console_handler)
        self.logger.addHandler(file_handler)

    def setup_output_directory(self):
        """Create output directory structure."""
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'models'), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, 'metrics'), exist_ok=True)
        self.logger.info(f"Output directory created: {self.output_dir}")

    def load_data(self) -> pd.DataFrame:
        """Load and validate preprocessed dataset."""
        self.logger.info(f"Loading data from {self.data_path}")

        try:
            df = pd.read_csv(self.data_path)
            self.logger.info(f"Data loaded successfully. Shape: {df.shape}")
            self.logger.info(f"Columns: {list(df.columns)}")
            self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

            # Convert date column to datetime
            df['date'] = pd.to_datetime(df['date'])

            # Sort by date to ensure chronological order
            df = df.sort_values('date').reset_index(drop=True)

            return df

        except Exception as e:
            self.logger.error(f"Error loading data: {str(e)}")
            raise

    def create_bilstm_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create BiLSTM-specific features including lag features and rolling statistics.

        Args:
            df: Input dataframe

        Returns:
            DataFrame with BiLSTM-specific features
        """
        self.logger.info("Creating BiLSTM-specific features")

        df = df.copy()

        # Temporal features from date
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week
        df['year'] = df['date'].dt.year

        # Cyclical encoding for temporal features (important for bidirectional patterns)
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

        # Lag features for arrivals (captures historical patterns)
        for lag in [1, 7, 14, 30]:
            df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

        # Rolling statistics (bidirectional context benefits from these)
        for window in [7, 14, 30]:
            df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(window=window, min_periods=1).mean()
            df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(window=window, min_periods=1).std()

        # Fill NaN values from lag features with forward fill then backward fill
        df = df.fillna(method='ffill').fillna(method='bfill')

        self.logger.info(f"BiLSTM features created. New shape: {df.shape}")

        return df

    def prepare_sequences(self, data: np.ndarray, sequence_length: int) -> np.ndarray:
        """
        Prepare sequences for BiLSTM input.

        Args:
            data: Input data array (n_samples, n_features)
            sequence_length: Length of sequences (lookback window)

        Returns:
            Array of sequences (n_sequences, sequence_length, n_features)
        """
        sequences = []
        for i in range(len(data) - sequence_length + 1):
            sequences.append(data[i:i + sequence_length])

        return np.array(sequences)

    def timeseries_train_test_split(self, df: pd.DataFrame,
                                   train_ratio: float = 0.7,
                                   val_ratio: float = 0.15,
                                   test_ratio: float = 0.15,
                                   sequence_length: int = 30) -> Tuple:
        """
        Perform time-series aware train/validation/test split with sequence creation.

        Args:
            df: Input dataframe
            train_ratio: Proportion for training set
            val_ratio: Proportion for validation set
            test_ratio: Proportion for test set
            sequence_length: Lookback window for BiLSTM

        Returns:
            Tuple of train, validation, and test sets
        """
        self.logger.info(f"Performing time-series split: {train_ratio}/{val_ratio}/{test_ratio}")
        self.logger.info(f"Sequence length (lookback): {sequence_length}")

        # Verify ratios sum to 1
        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

        n = len(df)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        # Split data
        train_df = df.iloc[:train_end].copy()
        val_df = df.iloc[train_end:val_end].copy()
        test_df = df.iloc[val_end:].copy()

        self.logger.info(f"Train set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})")
        self.logger.info(f"Validation set: {len(val_df)} samples ({val_df['date'].min()} to {val_df['date'].max()})")
        self.logger.info(f"Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})")

        # Store dates
        self.train_dates = train_df['date'].values
        self.val_dates = val_df['date'].values
        self.test_dates = test_df['date'].values

        # Define feature columns (exclude date and target)
        feature_cols = [col for col in df.columns if col not in ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']]

        # Extract features and target
        X_train = train_df[feature_cols].values
        X_val = val_df[feature_cols].values
        X_test = test_df[feature_cols].values

        y_train = train_df['arrivals'].values
        y_val = val_df['arrivals'].values
        y_test = test_df['arrivals'].values

        # Scale features
        self.logger.info("Scaling features using StandardScaler")
        self.scaler_X = StandardScaler()
        X_train_scaled = self.scaler_X.fit_transform(X_train)
        X_val_scaled = self.scaler_X.transform(X_val)
        X_test_scaled = self.scaler_X.transform(X_test)

        # Scale target
        self.logger.info("Scaling target using MinMaxScaler")
        self.scaler_y = MinMaxScaler()
        y_train_scaled = self.scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val_scaled = self.scaler_y.transform(y_val.reshape(-1, 1)).flatten()
        y_test_scaled = self.scaler_y.transform(y_test.reshape(-1, 1)).flatten()

        # Create sequences for BiLSTM
        self.logger.info("Creating sequences for BiLSTM")
        X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
        X_val_seq = self.prepare_sequences(X_val_scaled, sequence_length)
        X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)

        # Align y with sequences (remove first sequence_length-1 samples)
        y_train_seq = y_train_scaled[sequence_length - 1:]
        y_val_seq = y_val_scaled[sequence_length - 1:]
        y_test_seq = y_test_scaled[sequence_length - 1:]

        self.logger.info(f"Final shapes - X_train: {X_train_seq.shape}, y_train: {y_train_seq.shape}")
        self.logger.info(f"Final shapes - X_val: {X_val_seq.shape}, y_val: {y_val_seq.shape}")
        self.logger.info(f"Final shapes - X_test: {X_test_seq.shape}, y_test: {y_test_seq.shape}")

        return (X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq)

    def build_bilstm_model(self, input_shape: Tuple[int, int],
                          lstm_units: List[int] = [64, 32],
                          dropout_rate: float = 0.2,
                          learning_rate: float = 0.001) -> keras.Model:
        """
        Build Bidirectional LSTM model architecture.

        BiLSTM processes sequences in both forward and backward directions,
        effectively doubling the number of parameters compared to unidirectional LSTM.

        Args:
            input_shape: Shape of input (sequence_length, n_features)
            lstm_units: List of units for each BiLSTM layer (note: actual params = 2x due to bidirectionality)
            dropout_rate: Dropout rate for regularization
            learning_rate: Learning rate for optimizer

        Returns:
            Compiled Keras model
        """
        model = Sequential(name='BiLSTM_Tourism_Predictor')

        # First Bidirectional LSTM layer
        model.add(Bidirectional(
            LSTM(units=lstm_units[0],
                 return_sequences=len(lstm_units) > 1,
                 name='lstm_1'),
            input_shape=input_shape,
            name='bilstm_1'
        ))
        model.add(BatchNormalization(name='batch_norm_1'))
        model.add(Dropout(dropout_rate, name='dropout_1'))

        # Additional Bidirectional LSTM layers
        for i, units in enumerate(lstm_units[1:], start=2):
            return_seq = i < len(lstm_units)
            model.add(Bidirectional(
                LSTM(units=units,
                     return_sequences=return_seq,
                     name=f'lstm_{i}'),
                name=f'bilstm_{i}'
            ))
            model.add(BatchNormalization(name=f'batch_norm_{i}'))
            model.add(Dropout(dropout_rate, name=f'dropout_{i}'))

        # Dense layers
        model.add(Dense(32, activation='relu', name='dense_1'))
        model.add(Dropout(dropout_rate, name='dropout_final'))
        model.add(Dense(1, activation='linear', name='output'))

        # Compile model
        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

        return model

    def hyperparameter_tuning(self, X_train: np.ndarray, y_train: np.ndarray,
                            X_val: np.ndarray, y_val: np.ndarray) -> Dict[str, Any]:
        """
        Perform hyperparameter tuning using grid search.

        Note: BiLSTM units are smaller than LSTM units because bidirectionality
        doubles the effective parameters.

        Args:
            X_train: Training features
            y_train: Training target
            X_val: Validation features
            y_val: Validation target

        Returns:
            Dictionary of best hyperparameters
        """
        self.logger.info("Starting hyperparameter tuning for BiLSTM")
        self.logger.info("Note: BiLSTM units are halved compared to LSTM due to bidirectionality")

        # Define hyperparameter search space (smaller units due to bidirectionality)
        param_grid = {
            'lstm_units': [[64, 32], [128, 64], [64, 32, 16]],
            'dropout_rate': [0.2, 0.3],
            'learning_rate': [0.001, 0.0005],
            'batch_size': [32, 64]
        }

        best_val_loss = float('inf')
        best_params = None

        total_combinations = (len(param_grid['lstm_units']) *
                            len(param_grid['dropout_rate']) *
                            len(param_grid['learning_rate']) *
                            len(param_grid['batch_size']))

        self.logger.info(f"Testing {total_combinations} hyperparameter combinations")

        trial = 0
        for lstm_units in param_grid['lstm_units']:
            for dropout_rate in param_grid['dropout_rate']:
                for learning_rate in param_grid['learning_rate']:
                    for batch_size in param_grid['batch_size']:
                        trial += 1

                        self.logger.info(f"\nTrial {trial}/{total_combinations}")
                        self.logger.info(f"Params: lstm_units={lstm_units}, dropout={dropout_rate}, "
                                       f"lr={learning_rate}, batch_size={batch_size}")
                        self.logger.info(f"  (Effective BiLSTM params: {[u*2 for u in lstm_units]})")

                        # Build model
                        model = self.build_bilstm_model(
                            input_shape=(X_train.shape[1], X_train.shape[2]),
                            lstm_units=lstm_units,
                            dropout_rate=dropout_rate,
                            learning_rate=learning_rate
                        )

                        # Callbacks
                        early_stop = EarlyStopping(monitor='val_loss', patience=10,
                                                  restore_best_weights=True, verbose=0)
                        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                                     patience=5, min_lr=1e-7, verbose=0)

                        # Train
                        history = model.fit(
                            X_train, y_train,
                            validation_data=(X_val, y_val),
                            epochs=100,
                            batch_size=batch_size,
                            callbacks=[early_stop, reduce_lr],
                            verbose=0
                        )

                        # Evaluate
                        val_loss = min(history.history['val_loss'])
                        self.logger.info(f"Validation loss: {val_loss:.6f}")

                        # Update best parameters
                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            best_params = {
                                'lstm_units': lstm_units,
                                'dropout_rate': dropout_rate,
                                'learning_rate': learning_rate,
                                'batch_size': batch_size,
                                'val_loss': val_loss
                            }
                            self.logger.info(f"*** New best parameters found! Val loss: {val_loss:.6f} ***")

        self.logger.info(f"\nHyperparameter tuning completed")
        self.logger.info(f"Best parameters: {best_params}")
        self.best_params = best_params

        # Save best parameters
        with open(os.path.join(self.output_dir, 'best_hyperparameters.json'), 'w') as f:
            json.dump(best_params, f, indent=4)

        return best_params

    def train_final_model(self, X_train: np.ndarray, y_train: np.ndarray,
                         X_val: np.ndarray, y_val: np.ndarray,
                         params: Dict[str, Any]) -> keras.Model:
        """
        Train final BiLSTM model with best hyperparameters.

        Args:
            X_train: Training features
            y_train: Training target
            X_val: Validation features
            y_val: Validation target
            params: Best hyperparameters

        Returns:
            Trained Keras model
        """
        self.logger.info("Training final BiLSTM model with best hyperparameters")

        # Build model
        model = self.build_bilstm_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            lstm_units=params['lstm_units'],
            dropout_rate=params['dropout_rate'],
            learning_rate=params['learning_rate']
        )

        # Print model summary
        self.logger.info("\nBiLSTM Model Architecture:")
        model.summary(print_fn=lambda x: self.logger.info(x))

        # Callbacks
        model_path = os.path.join(self.output_dir, 'models', 'best_bilstm_model.h5')
        checkpoint = ModelCheckpoint(model_path, monitor='val_loss',
                                    save_best_only=True, verbose=1)
        early_stop = EarlyStopping(monitor='val_loss', patience=20,
                                  restore_best_weights=True, verbose=1)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=7, min_lr=1e-7, verbose=1)

        # Train
        self.logger.info("Starting training...")
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=200,
            batch_size=params['batch_size'],
            callbacks=[checkpoint, early_stop, reduce_lr],
            verbose=1
        )

        self.history = history.history
        self.best_model = model

        self.logger.info("Training completed")

        return model

    def calculate_metrics(self, y_true: np.ndarray, y_pred: np.ndarray,
                         set_name: str = '') -> Dict[str, float]:
        """
        Calculate evaluation metrics.

        Args:
            y_true: True values
            y_pred: Predicted values
            set_name: Name of dataset (train/val/test)

        Returns:
            Dictionary of metrics
        """
        # Inverse transform predictions and true values
        y_true_original = self.scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
        y_pred_original = self.scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

        # Calculate metrics
        mse = mean_squared_error(y_true_original, y_pred_original)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true_original, y_pred_original)
        mape = mean_absolute_percentage_error(y_true_original, y_pred_original) * 100

        metrics = {
            'MSE': float(mse),
            'RMSE': float(rmse),
            'R2': float(r2),
            'MAPE': float(mape)
        }

        self.logger.info(f"\n{set_name} Metrics:")
        for metric, value in metrics.items():
            self.logger.info(f"  {metric}: {value:.4f}")

        return metrics

    def time_series_cross_validation(self, df: pd.DataFrame,
                                     n_splits: int = 5,
                                     sequence_length: int = 30) -> List[Dict[str, float]]:
        """
        Perform time-series cross-validation with BiLSTM.

        Args:
            df: Input dataframe
            n_splits: Number of CV splits
            sequence_length: Lookback window

        Returns:
            List of metrics for each fold
        """
        self.logger.info(f"\nPerforming time-series cross-validation with {n_splits} splits")

        feature_cols = [col for col in df.columns if col not in ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']]

        n = len(df)
        fold_size = n // (n_splits + 1)

        cv_metrics = []

        for fold in range(n_splits):
            self.logger.info(f"\n--- Fold {fold + 1}/{n_splits} ---")

            # Define train/test split for this fold
            test_start = fold_size * (fold + 1)
            test_end = test_start + fold_size

            train_df = df.iloc[:test_start].copy()
            test_df = df.iloc[test_start:test_end].copy()

            if len(train_df) < sequence_length or len(test_df) < sequence_length:
                self.logger.warning(f"Fold {fold + 1} skipped due to insufficient data")
                continue

            # Prepare data
            X_train = train_df[feature_cols].values
            X_test = test_df[feature_cols].values
            y_train = train_df['arrivals'].values
            y_test = test_df['arrivals'].values

            # Scale
            scaler_X = StandardScaler()
            scaler_y = MinMaxScaler()

            X_train_scaled = scaler_X.fit_transform(X_train)
            X_test_scaled = scaler_X.transform(X_test)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

            # Create sequences
            X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
            X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)
            y_train_seq = y_train_scaled[sequence_length - 1:]
            y_test_seq = y_test_scaled[sequence_length - 1:]

            # Build and train model (using best params if available)
            if self.best_params:
                params = self.best_params
            else:
                params = {
                    'lstm_units': [64, 32],
                    'dropout_rate': 0.2,
                    'learning_rate': 0.001,
                    'batch_size': 32
                }

            model = self.build_bilstm_model(
                input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
                lstm_units=params['lstm_units'],
                dropout_rate=params['dropout_rate'],
                learning_rate=params['learning_rate']
            )

            early_stop = EarlyStopping(monitor='loss', patience=10,
                                      restore_best_weights=True, verbose=0)

            model.fit(X_train_seq, y_train_seq,
                     epochs=50,
                     batch_size=params['batch_size'],
                     callbacks=[early_stop],
                     verbose=0)

            # Predict and evaluate
            y_pred = model.predict(X_test_seq, verbose=0).flatten()

            # Calculate metrics
            y_true_original = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
            y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

            fold_metrics = {
                'fold': fold + 1,
                'MSE': float(mean_squared_error(y_true_original, y_pred_original)),
                'RMSE': float(np.sqrt(mean_squared_error(y_true_original, y_pred_original))),
                'R2': float(r2_score(y_true_original, y_pred_original)),
                'MAPE': float(mean_absolute_percentage_error(y_true_original, y_pred_original) * 100)
            }

            cv_metrics.append(fold_metrics)

            self.logger.info(f"Fold {fold + 1} metrics:")
            for metric, value in fold_metrics.items():
                if metric != 'fold':
                    self.logger.info(f"  {metric}: {value:.4f}")

        # Calculate average metrics
        if cv_metrics:
            avg_metrics = {
                'MSE': np.mean([m['MSE'] for m in cv_metrics]),
                'RMSE': np.mean([m['RMSE'] for m in cv_metrics]),
                'R2': np.mean([m['R2'] for m in cv_metrics]),
                'MAPE': np.mean([m['MAPE'] for m in cv_metrics])
            }

            self.logger.info(f"\nAverage CV Metrics:")
            for metric, value in avg_metrics.items():
                self.logger.info(f"  {metric}: {value:.4f}")

        return cv_metrics

    def run_full_pipeline(self, sequence_length: int = 30,
                         perform_cv: bool = True):
        """
        Execute the complete BiLSTM model exploration pipeline.

        Args:
            sequence_length: Lookback window for BiLSTM
            perform_cv: Whether to perform cross-validation
        """
        self.logger.info("="*80)
        self.logger.info("STARTING BiLSTM MODEL EXPLORATION PIPELINE")
        self.logger.info("="*80)

        # 1. Load Data
        self.logger.info("\n[STEP 1] Loading Data")
        df = self.load_data()

        # 2. Create BiLSTM Features
        self.logger.info("\n[STEP 2] Creating BiLSTM-Specific Features")
        df = self.create_bilstm_features(df)

        # 3. Train/Val/Test Split
        self.logger.info("\n[STEP 3] Splitting Data (70/15/15)")
        X_train, y_train, X_val, y_val, X_test, y_test = self.timeseries_train_test_split(
            df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15,
            sequence_length=sequence_length
        )

        # Store splits
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test

        # 4. Hyperparameter Tuning
        self.logger.info("\n[STEP 4] Hyperparameter Tuning")
        best_params = self.hyperparameter_tuning(X_train, y_train, X_val, y_val)

        # 5. Train Final Model
        self.logger.info("\n[STEP 5] Training Final BiLSTM Model")
        model = self.train_final_model(X_train, y_train, X_val, y_val, best_params)

        # 6. Evaluation
        self.logger.info("\n[STEP 6] Model Evaluation")

        # Predictions
        y_train_pred = model.predict(X_train, verbose=0).flatten()
        y_val_pred = model.predict(X_val, verbose=0).flatten()
        y_test_pred = model.predict(X_test, verbose=0).flatten()

        # Calculate metrics
        train_metrics = self.calculate_metrics(y_train, y_train_pred, 'Training Set')
        val_metrics = self.calculate_metrics(y_val, y_val_pred, 'Validation Set')
        test_metrics = self.calculate_metrics(y_test, y_test_pred, 'Test Set')

        # 7. Cross-Validation
        cv_metrics = None
        if perform_cv:
            self.logger.info("\n[STEP 7] Time-Series Cross-Validation")
            cv_metrics = self.time_series_cross_validation(df, n_splits=5,
                                                          sequence_length=sequence_length)

        # 8. Save Results
        self.logger.info("\n[STEP 8] Saving Results")

        all_metrics = {
            'model_type': 'BiLSTM',
            'train_metrics': train_metrics,
            'validation_metrics': val_metrics,
            'test_metrics': test_metrics,
            'cv_metrics': cv_metrics if cv_metrics else [],
            'best_hyperparameters': best_params,
            'training_history': {
                'loss': [float(x) for x in self.history['loss']],
                'val_loss': [float(x) for x in self.history['val_loss']]
            }
        }

        metrics_path = os.path.join(self.output_dir, 'metrics', 'all_metrics.json')
        with open(metrics_path, 'w') as f:
            json.dump(all_metrics, f, indent=4)

        self.logger.info(f"All metrics saved to {metrics_path}")

        # Save predictions
        predictions_df = pd.DataFrame({
            'y_true': self.scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten(),
            'y_pred': self.scaler_y.inverse_transform(y_test_pred.reshape(-1, 1)).flatten()
        })
        pred_path = os.path.join(self.output_dir, 'test_predictions.csv')
        predictions_df.to_csv(pred_path, index=False)
        self.logger.info(f"Test predictions saved to {pred_path}")

        self.logger.info("\n" + "="*80)
        self.logger.info("BiLSTM MODEL EXPLORATION PIPELINE COMPLETED SUCCESSFULLY")
        self.logger.info("="*80)

        return all_metrics


def main():
    """Main execution function."""
    # Configuration
    DATA_PATH = 'preprocessed-dataset.csv'
    OUTPUT_DIR = 'bilstm_output'
    SEQUENCE_LENGTH = 30  # 30-day lookback window
    PERFORM_CV = True

    print("="*80)
    print("BiLSTM Model Explorer for Sri Lankan Tourism Arrivals")
    print("="*80)
    print("\nBidirectional LSTM processes sequences in both directions,")
    print("capturing dependencies from past and future context.")
    print("\nConfiguration:")
    print(f"  Data Path: {DATA_PATH}")
    print(f"  Output Directory: {OUTPUT_DIR}")
    print(f"  Sequence Length: {SEQUENCE_LENGTH} days")
    print(f"  Cross-Validation: {PERFORM_CV}")
    print("="*80)

    # Initialize explorer
    explorer = BiLSTMModelExplorer(data_path=DATA_PATH, output_dir=OUTPUT_DIR)

    # Run pipeline
    metrics = explorer.run_full_pipeline(
        sequence_length=SEQUENCE_LENGTH,
        perform_cv=PERFORM_CV
    )

    print("\n" + "="*80)
    print("EXECUTION COMPLETED")
    print("="*80)
    print(f"\nFinal Test Set Metrics:")
    for metric, value in metrics['test_metrics'].items():
        print(f"  {metric}: {value:.4f}")

    print(f"\nOutputs saved to: {OUTPUT_DIR}/")
    print("  - models/best_bilstm_model.h5")
    print("  - metrics/all_metrics.json")
    print("  - best_hyperparameters.json")
    print("  - test_predictions.csv")


if __name__ == '__main__':
    main()

2026-03-18 04:33:55,779 - BiLSTMModelExplorer - INFO - Output directory created: bilstm_output
INFO:BiLSTMModelExplorer:Output directory created: bilstm_output
2026-03-18 04:33:55,781 - BiLSTMModelExplorer - INFO - BiLSTM Model Explorer initialized successfully
INFO:BiLSTMModelExplorer:BiLSTM Model Explorer initialized successfully
2026-03-18 04:33:55,783 - BiLSTMModelExplorer - INFO - ================================================================================
INFO:BiLSTMModelExplorer:================================================================================
2026-03-18 04:33:55,785 - BiLSTMModelExplorer - INFO - STARTING BiLSTM MODEL EXPLORATION PIPELINE
INFO:BiLSTMModelExplorer:STARTING BiLSTM MODEL EXPLORATION PIPELINE
2026-03-18 04:33:55,787 - BiLSTMModelExplorer - INFO - ================================================================================
INFO:BiLSTMModelExplorer:================================================================================
2026-03-18 04:33

BiLSTM Model Explorer for Sri Lankan Tourism Arrivals

Bidirectional LSTM processes sequences in both directions,
capturing dependencies from past and future context.

Configuration:
  Data Path: preprocessed-dataset.csv
  Output Directory: bilstm_output
  Sequence Length: 30 days
  Cross-Validation: True


2026-03-18 04:33:55,812 - BiLSTMModelExplorer - INFO - Data loaded successfully. Shape: (5740, 20)
INFO:BiLSTMModelExplorer:Data loaded successfully. Shape: (5740, 20)
2026-03-18 04:33:55,814 - BiLSTMModelExplorer - INFO - Columns: ['date', 'brent_crude_price', 'cny_lkr', 'covid_impact_factor', 'crisis_impact_factor', 'eur_lkr', 'event_encoded', 'gbp_lkr', 'gdp_per_capita', 'humidity', 'image_search', 'inflation_rate', 'inr_lkr', 'outlier_flag', 'precipitation', 'rub_lkr', 'temperature', 'usd_lkr', 'web_search', 'arrivals']
INFO:BiLSTMModelExplorer:Columns: ['date', 'brent_crude_price', 'cny_lkr', 'covid_impact_factor', 'crisis_impact_factor', 'eur_lkr', 'event_encoded', 'gbp_lkr', 'gdp_per_capita', 'humidity', 'image_search', 'inflation_rate', 'inr_lkr', 'outlier_flag', 'precipitation', 'rub_lkr', 'temperature', 'usd_lkr', 'web_search', 'arrivals']
2026-03-18 04:33:55,819 - BiLSTMModelExplorer - INFO - Date range: 2010-01-01 to 2025-09-18
INFO:BiLSTMModelExplorer:Date range: 2010-01-0

2026-03-18 05:03:52,667 - BiLSTMModelExplorer - INFO - Model: "BiLSTM_Tourism_Predictor"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bilstm_1 (Bidirectional)        │ (None, 30, 256)        │       169,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 30, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_2 (Bidirectional)        │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼────

Epoch 1/200
124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.8718 - mae: 0.7229
Epoch 1: val_loss improved from None to 0.02519, saving model to bilstm_output/models/best_bilstm_model.h5


...


Epoch 29: finished saving model to bilstm_output/models/best_bilstm_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0046 - mae: 0.0514 - val_loss: 0.0059 - val_mae: 0.0588 - learning_rate: 5.0000e-04
Epoch 30/200
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0047 - mae: 0.0516
Epoch 30: val_loss improved from 0.00594 to 0.00555, saving model to bilstm_output/models/best_bilstm_model.h5



Epoch 30: finished saving model to bilstm_output/models/best_bilstm_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.0046 - mae: 0.0512 - val_loss: 0.0055 - val_mae: 0.0572 - learning_rate: 5.0000e-04
Epoch 31/200
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0046 - mae: 0.0514
Epoch 31: val_loss did not improve from 0.00555
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - loss: 0.0044 - mae: 0.0503 - val_loss: 0.0057 - val_mae: 0.0561 - learning_rate: 5.0000e-04
Epoch 32/200
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0048 - mae: 0.0516
Epoch 32: val_loss did not improve from 0.00555
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0046 - mae: 0.0503 - val_loss: 0.0061 - val_mae: 0.0599 - learning_rate: 5.0000e-04
Epoch 33/200
122/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0042 - mae: 0.0483
Epoch 33: val_loss did not improve from 0.00555
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0043 - mae: 0.0490 - val_loss: 0.0062 - val_mae: 0.0594 - learning


Epoch 37: finished saving model to bilstm_output/models/best_bilstm_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.0040 - mae: 0.0472 - val_loss: 0.0047 - val_mae: 0.0534 - learning_rate: 5.0000e-04
Epoch 38/200
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0038 - mae: 0.0467
Epoch 38: val_loss did not improve from 0.00473
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - loss: 0.0038 - mae: 0.0464 - val_loss: 0.0059 - val_mae: 0.0604 - learning_rate: 5.0000e-04
Epoch 39/200
124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0037 - mae: 0.0461
Epoch 39: val_loss did not improve from 0.00473
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0037 - mae: 0.0455 - val_loss: 0.0060 - val_mae: 0.0582 - learning_rate: 5.0000e-04
Epoch 40/200
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0035 - mae: 0.0452
Epoch 40: val_loss did not improve from 0.00473
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0035 - mae: 0.0447 - val_loss: 0.0055 - val_mae: 0.0579 - learning

2026-03-18 05:05:58,257 - BiLSTMModelExplorer - INFO - Training completed
INFO:BiLSTMModelExplorer:Training completed
2026-03-18 05:05:58,258 - BiLSTMModelExplorer - INFO - 
[STEP 6] Model Evaluation
INFO:BiLSTMModelExplorer:
[STEP 6] Model Evaluation
2026-03-18 05:05:59,972 - BiLSTMModelExplorer - INFO - 
Training Set Metrics:
INFO:BiLSTMModelExplorer:
Training Set Metrics:
2026-03-18 05:05:59,976 - BiLSTMModelExplorer - INFO -   MSE: 256862.0387
INFO:BiLSTMModelExplorer:  MSE: 256862.0387
2026-03-18 05:05:59,979 - BiLSTMModelExplorer - INFO -   RMSE: 506.8156
INFO:BiLSTMModelExplorer:  RMSE: 506.8156
2026-03-18 05:05:59,980 - BiLSTMModelExplorer - INFO -   R2: 0.9525
INFO:BiLSTMModelExplorer:  R2: 0.9525
2026-03-18 05:05:59,982 - BiLSTMModelExplorer - INFO -   MAPE: 10225798232523544576.0000
INFO:BiLSTMModelExplorer:  MAPE: 10225798232523544576.0000
2026-03-18 05:05:59,987 - BiLSTMModelExplorer - INFO - 
Validation Set Metrics:
INFO:BiLSTMModelExplorer:
Validation Set Metrics:
2026-0


EXECUTION COMPLETED

Final Test Set Metrics:
  MSE: 5474035.2282
  RMSE: 2339.6656
  R2: -0.2482
  MAPE: 33.6980

Outputs saved to: bilstm_output/
  - models/best_bilstm_model.h5
  - metrics/all_metrics.json
  - best_hyperparameters.json
  - test_predictions.csv


In [ ]:
"""
GRU Model Exploration for Sri Lankan Tourism Arrivals Prediction
=================================================================
This module implements a production-ready GRU model with hyperparameter tuning,
time-series aware validation, and comprehensive evaluation metrics.

It is structurally aligned with:
- lstm_model_explorer.py
- bilstm_model_explorer.py

Author: ML Research Team
Date: December 2025
"""

import numpy as np
import pandas as pd
import logging
import json
import warnings
from datetime import datetime
from typing import Tuple, Dict, List, Any
import os

# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


class GRUModelExplorer:
    """
    GRU Model Explorer for time-series forecasting with comprehensive
    preprocessing, hyperparameter tuning, and evaluation.

    GRU (Gated Recurrent Unit) is a lighter alternative to LSTM, often with
    similar performance but fewer parameters and faster training.
    """

    def __init__(self, data_path: str, output_dir: str = "gru_output"):
        """
        Initialize the GRU Model Explorer.

        Args:
            data_path: Path to preprocessed dataset CSV
            output_dir: Directory for saving outputs
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.setup_logging()
        self.setup_output_directory()

        # Model artifacts
        self.scaler_X = None
        self.scaler_y = None
        self.best_model = None
        self.best_params = None
        self.history = None

        # Data splits
        self.X_train, self.X_val, self.X_test = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.train_dates, self.val_dates, self.test_dates = None, None, None

        self.logger.info("GRU Model Explorer initialized successfully")

    # -------------------------------------------------------------------------
    # Logging & IO
    # -------------------------------------------------------------------------
    def setup_logging(self):
        """Configure logging with both file and console handlers."""
        log_format = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"

        self.logger = logging.getLogger("GRUModelExplorer")
        self.logger.setLevel(logging.INFO)
        self.logger.handlers = []

        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter(log_format))

        file_handler = logging.FileHandler(
            f"gru_exploration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        )
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(logging.Formatter(log_format))

        self.logger.addHandler(console_handler)
        self.logger.addHandler(file_handler)

    def setup_output_directory(self):
        """Create output directory structure."""
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "models"), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "metrics"), exist_ok=True)
        self.logger.info(f"Output directory created: {self.output_dir}")

    def load_data(self) -> pd.DataFrame:
        """Load and validate preprocessed dataset."""
        self.logger.info(f"Loading data from {self.data_path}")

        try:
            df = pd.read_csv(self.data_path)
            self.logger.info(f"Data loaded successfully. Shape: {df.shape}")
            self.logger.info(f"Columns: {list(df.columns)}")
            self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)

            return df

        except Exception as e:
            self.logger.error(f"Error loading data: {str(e)}")
            raise

    # -------------------------------------------------------------------------
    # Feature Engineering & Sequences
    # -------------------------------------------------------------------------
    def create_gru_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create GRU-specific features including lag features and rolling statistics.

        Args:
            df: Input dataframe

        Returns:
            DataFrame with GRU-specific features
        """
        self.logger.info("Creating GRU-specific features")

        df = df.copy()

        # Temporal features
        df["day_of_week"] = df["date"].dt.dayofweek
        df["day_of_month"] = df["date"].dt.day
        df["month"] = df["date"].dt.month
        df["quarter"] = df["date"].dt.quarter
        df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
        df["year"] = df["date"].dt.year

        # Cyclical encoding
        df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
        df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

        # Lag features for arrivals
        for lag in [1, 7, 14, 30]:
            df[f"arrivals_lag_{lag}"] = df["arrivals"].shift(lag)

        # Rolling statistics
        for window in [7, 14, 30]:
            df[f"arrivals_rolling_mean_{window}"] = df["arrivals"].rolling(
                window=window, min_periods=1
            ).mean()
            df[f"arrivals_rolling_std_{window}"] = df["arrivals"].rolling(
                window=window, min_periods=1
            ).std()

        # Fill NaNs from lags/rolling
        df = df.fillna(method="ffill").fillna(method="bfill")

        self.logger.info(f"GRU features created. New shape: {df.shape}")
        return df

    def prepare_sequences(self, data: np.ndarray, sequence_length: int) -> np.ndarray:
        """
        Prepare sequences for GRU input.

        Args:
            data: Input data array (n_samples, n_features)
            sequence_length: Length of sequences (lookback window)

        Returns:
            Array of sequences (n_sequences, sequence_length, n_features)
        """
        sequences = []
        for i in range(len(data) - sequence_length + 1):
            sequences.append(data[i : i + sequence_length])

        return np.array(sequences)

    # -------------------------------------------------------------------------
    # Train/Val/Test Split
    # -------------------------------------------------------------------------
    def timeseries_train_test_split(
        self,
        df: pd.DataFrame,
        train_ratio: float = 0.7,
        val_ratio: float = 0.15,
        test_ratio: float = 0.15,
        sequence_length: int = 30,
    ) -> Tuple:
        """
        Perform time-series aware train/validation/test split with sequence creation.

        Args:
            df: Input dataframe
            train_ratio: Proportion for training set
            val_ratio: Proportion for validation set
            test_ratio: Proportion for test set
            sequence_length: Lookback window for GRU

        Returns:
            Tuple of train, validation, and test sets
        """
        self.logger.info(
            f"Performing time-series split: {train_ratio}/{val_ratio}/{test_ratio}"
        )
        self.logger.info(f"Sequence length (lookback): {sequence_length}")

        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"

        n = len(df)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))

        train_df = df.iloc[:train_end].copy()
        val_df = df.iloc[train_end:val_end].copy()
        test_df = df.iloc[val_end:].copy()

        self.logger.info(
            f"Train set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})"
        )
        self.logger.info(
            f"Validation set: {len(val_df)} samples ({val_df['date'].min()} to {val_df['date'].max()})"
        )
        self.logger.info(
            f"Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})"
        )

        self.train_dates = train_df["date"].values
        self.val_dates = val_df["date"].values
        self.test_dates = test_df["date"].values

        feature_cols = [
            col
            for col in df.columns
            if col not in ["date", "arrivals", "arrivals_robust_scaled", "outlier_flag"]
        ]

        X_train = train_df[feature_cols].values
        X_val = val_df[feature_cols].values
        X_test = test_df[feature_cols].values

        y_train = train_df["arrivals"].values
        y_val = val_df["arrivals"].values
        y_test = test_df["arrivals"].values

        # Scaling
        self.logger.info("Scaling features using StandardScaler")
        self.scaler_X = StandardScaler()
        X_train_scaled = self.scaler_X.fit_transform(X_train)
        X_val_scaled = self.scaler_X.transform(X_val)
        X_test_scaled = self.scaler_X.transform(X_test)

        self.logger.info("Scaling target using MinMaxScaler")
        self.scaler_y = MinMaxScaler()
        y_train_scaled = self.scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val_scaled = self.scaler_y.transform(y_val.reshape(-1, 1)).flatten()
        y_test_scaled = self.scaler_y.transform(y_test.reshape(-1, 1)).flatten()

        # Sequences
        self.logger.info("Creating sequences for GRU")
        X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
        X_val_seq = self.prepare_sequences(X_val_scaled, sequence_length)
        X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)

        y_train_seq = y_train_scaled[sequence_length - 1 :]
        y_val_seq = y_val_scaled[sequence_length - 1 :]
        y_test_seq = y_test_scaled[sequence_length - 1 :]

        self.logger.info(
            f"Final shapes - X_train: {X_train_seq.shape}, y_train: {y_train_seq.shape}"
        )
        self.logger.info(
            f"Final shapes - X_val: {X_val_seq.shape}, y_val: {y_val_seq.shape}"
        )
        self.logger.info(
            f"Final shapes - X_test: {X_test_seq.shape}, y_test: {y_test_seq.shape}"
        )

        return X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq

    # -------------------------------------------------------------------------
    # GRU Architecture & Training
    # -------------------------------------------------------------------------
    def build_gru_model(
        self,
        input_shape: Tuple[int, int],
        gru_units: List[int] = [128, 64],
        dropout_rate: float = 0.2,
        learning_rate: float = 0.001,
    ) -> keras.Model:
        """
        Build GRU model architecture.

        Args:
            input_shape: Shape of input (sequence_length, n_features)
            gru_units: List of units for each GRU layer
            dropout_rate: Dropout rate for regularization
            learning_rate: Learning rate for optimizer

        Returns:
            Compiled Keras model
        """
        model = Sequential(name="GRU_Tourism_Predictor")

        # First GRU layer
        model.add(
            GRU(
                units=gru_units[0],
                return_sequences=len(gru_units) > 1,
                input_shape=input_shape,
                name="gru_1",
            )
        )
        model.add(BatchNormalization(name="batch_norm_1"))
        model.add(Dropout(dropout_rate, name="dropout_1"))

        # Additional GRU layers
        for i, units in enumerate(gru_units[1:], start=2):
            return_seq = i < len(gru_units)
            model.add(
                GRU(
                    units=units,
                    return_sequences=return_seq,
                    name=f"gru_{i}",
                )
            )
            model.add(BatchNormalization(name=f"batch_norm_{i}"))
            model.add(Dropout(dropout_rate, name=f"dropout_{i}"))

        model.add(Dense(32, activation="relu", name="dense_1"))
        model.add(Dropout(dropout_rate, name="dropout_final"))
        model.add(Dense(1, activation="linear", name="output"))

        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

        return model

    def hyperparameter_tuning(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
    ) -> Dict[str, Any]:
        """
        Perform hyperparameter tuning using grid search.

        Args:
            X_train: Training features
            y_train: Training target
            X_val: Validation features
            y_val: Validation target

        Returns:
            Dictionary of best hyperparameters
        """
        self.logger.info("Starting hyperparameter tuning for GRU")

        param_grid = {
            "gru_units": [[128, 64], [256, 128], [128, 64, 32]],
            "dropout_rate": [0.2, 0.3],
            "learning_rate": [0.001, 0.0005],
            "batch_size": [32, 64],
        }

        best_val_loss = float("inf")
        best_params = None

        total_combinations = (
            len(param_grid["gru_units"])
            * len(param_grid["dropout_rate"])
            * len(param_grid["learning_rate"])
            * len(param_grid["batch_size"])
        )

        self.logger.info(f"Testing {total_combinations} hyperparameter combinations")

        trial = 0
        for gru_units in param_grid["gru_units"]:
            for dropout_rate in param_grid["dropout_rate"]:
                for learning_rate in param_grid["learning_rate"]:
                    for batch_size in param_grid["batch_size"]:
                        trial += 1

                        self.logger.info(f"\nTrial {trial}/{total_combinations}")
                        self.logger.info(
                            f"Params: gru_units={gru_units}, dropout={dropout_rate}, "
                            f"lr={learning_rate}, batch_size={batch_size}"
                        )

                        model = self.build_gru_model(
                            input_shape=(X_train.shape[1], X_train.shape[2]),
                            gru_units=gru_units,
                            dropout_rate=dropout_rate,
                            learning_rate=learning_rate,
                        )

                        early_stop = EarlyStopping(
                            monitor="val_loss", patience=10, restore_best_weights=True, verbose=0
                        )
                        reduce_lr = ReduceLROnPlateau(
                            monitor="val_loss",
                            factor=0.5,
                            patience=5,
                            min_lr=1e-7,
                            verbose=0,
                        )

                        history = model.fit(
                            X_train,
                            y_train,
                            validation_data=(X_val, y_val),
                            epochs=100,
                            batch_size=batch_size,
                            callbacks=[early_stop, reduce_lr],
                            verbose=0,
                        )

                        val_loss = float(min(history.history["val_loss"]))
                        self.logger.info(f"Validation loss: {val_loss:.6f}")

                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            best_params = {
                                "gru_units": gru_units,
                                "dropout_rate": dropout_rate,
                                "learning_rate": learning_rate,
                                "batch_size": batch_size,
                                "val_loss": val_loss,
                            }
                            self.logger.info(
                                f"*** New best parameters found! Val loss: {val_loss:.6f} ***"
                            )

        self.logger.info("\nHyperparameter tuning completed")
        self.logger.info(f"Best parameters: {best_params}")
        self.best_params = best_params

        with open(
            os.path.join(self.output_dir, "best_hyperparameters.json"), "w", encoding="utf-8"
        ) as f:
            json.dump(best_params, f, indent=4)

        return best_params

    def train_final_model(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        params: Dict[str, Any],
    ) -> keras.Model:
        """
        Train final GRU model with best hyperparameters.
        """
        self.logger.info("Training final GRU model with best hyperparameters")

        model = self.build_gru_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            gru_units=params["gru_units"],
            dropout_rate=params["dropout_rate"],
            learning_rate=params["learning_rate"],
        )

        self.logger.info("\nGRU Model Architecture:")
        model.summary(print_fn=lambda x: self.logger.info(x))

        model_path = os.path.join(self.output_dir, "models", "best_gru_model.h5")
        checkpoint = ModelCheckpoint(
            model_path, monitor="val_loss", save_best_only=True, verbose=1
        )
        early_stop = EarlyStopping(
            monitor="val_loss", patience=20, restore_best_weights=True, verbose=1
        )
        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=7, min_lr=1e-7, verbose=1
        )

        self.logger.info("Starting training...")
        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=200,
            batch_size=params["batch_size"],
            callbacks=[checkpoint, early_stop, reduce_lr],
            verbose=1,
        )

        self.history = history.history
        self.best_model = model

        self.logger.info("Training completed")
        return model

    # -------------------------------------------------------------------------
    # Evaluation & CV
    # -------------------------------------------------------------------------
    def calculate_metrics(
        self, y_true: np.ndarray, y_pred: np.ndarray, set_name: str = ""
    ) -> Dict[str, float]:
        """
        Calculate evaluation metrics (R2, RMSE, MSE, MAPE) on original scale.
        """
        y_true_original = self.scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
        y_pred_original = self.scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

        mse = mean_squared_error(y_true_original, y_pred_original)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true_original, y_pred_original)
        mape = mean_absolute_percentage_error(y_true_original, y_pred_original) * 100

        metrics = {
            "MSE": float(mse),
            "RMSE": float(rmse),
            "R2": float(r2),
            "MAPE": float(mape),
        }

        self.logger.info(f"\n{set_name} Metrics:")
        for metric, value in metrics.items():
            self.logger.info(f"  {metric}: {value:.4f}")

        return metrics

    def time_series_cross_validation(
        self, df: pd.DataFrame, n_splits: int = 5, sequence_length: int = 30
    ) -> List[Dict[str, float]]:
        """
        Perform time-series cross-validation with GRU.
        """
        self.logger.info(f"\nPerforming time-series cross-validation with {n_splits} splits")

        feature_cols = [
            col
            for col in df.columns
            if col not in ["date", "arrivals", "arrivals_robust_scaled", "outlier_flag"]
        ]

        n = len(df)
        fold_size = n // (n_splits + 1)
        cv_metrics = []

        for fold in range(n_splits):
            self.logger.info(f"\n--- Fold {fold + 1}/{n_splits} ---")

            test_start = fold_size * (fold + 1)
            test_end = test_start + fold_size

            train_df = df.iloc[:test_start].copy()
            test_df = df.iloc[test_start:test_end].copy()

            if len(train_df) < sequence_length or len(test_df) < sequence_length:
                self.logger.warning(f"Fold {fold + 1} skipped due to insufficient data")
                continue

            X_train = train_df[feature_cols].values
            X_test = test_df[feature_cols].values
            y_train = train_df["arrivals"].values
            y_test = test_df["arrivals"].values

            scaler_X = StandardScaler()
            scaler_y = MinMaxScaler()

            X_train_scaled = scaler_X.fit_transform(X_train)
            X_test_scaled = scaler_X.transform(X_test)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

            X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
            X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)
            y_train_seq = y_train_scaled[sequence_length - 1 :]
            y_test_seq = y_test_scaled[sequence_length - 1 :]

            if self.best_params:
                params = self.best_params
            else:
                params = {
                    "gru_units": [128, 64],
                    "dropout_rate": 0.2,
                    "learning_rate": 0.001,
                    "batch_size": 32,
                }

            model = self.build_gru_model(
                input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
                gru_units=params["gru_units"],
                dropout_rate=params["dropout_rate"],
                learning_rate=params["learning_rate"],
            )

            early_stop = EarlyStopping(
                monitor="loss", patience=10, restore_best_weights=True, verbose=0
            )

            model.fit(
                X_train_seq,
                y_train_seq,
                epochs=50,
                batch_size=params["batch_size"],
                callbacks=[early_stop],
                verbose=0,
            )

            y_pred = model.predict(X_test_seq, verbose=0).flatten()

            y_true_original = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
            y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()

            fold_metrics = {
                "fold": fold + 1,
                "MSE": float(mean_squared_error(y_true_original, y_pred_original)),
                "RMSE": float(
                    np.sqrt(mean_squared_error(y_true_original, y_pred_original))
                ),
                "R2": float(r2_score(y_true_original, y_pred_original)),
                "MAPE": float(
                    mean_absolute_percentage_error(y_true_original, y_pred_original) * 100
                ),
            }

            cv_metrics.append(fold_metrics)

            self.logger.info(f"Fold {fold + 1} metrics:")
            for metric, value in fold_metrics.items():
                if metric != "fold":
                    self.logger.info(f"  {metric}: {value:.4f}")

        if cv_metrics:
            avg_metrics = {
                "MSE": float(np.mean([m["MSE"] for m in cv_metrics])),
                "RMSE": float(np.mean([m["RMSE"] for m in cv_metrics])),
                "R2": float(np.mean([m["R2"] for m in cv_metrics])),
                "MAPE": float(np.mean([m["MAPE"] for m in cv_metrics])),
            }
            self.logger.info("\nAverage CV Metrics:")
            for metric, value in avg_metrics.items():
                self.logger.info(f"  {metric}: {value:.4f}")

        return cv_metrics

    # -------------------------------------------------------------------------
    # Full Pipeline
    # -------------------------------------------------------------------------
    def run_full_pipeline(self, sequence_length: int = 30, perform_cv: bool = True):
        """
        Execute the complete GRU model exploration pipeline.
        """
        self.logger.info("=" * 80)
        self.logger.info("STARTING GRU MODEL EXPLORATION PIPELINE")
        self.logger.info("=" * 80)

        # 1. Load Data
        self.logger.info("\n[STEP 1] Loading Data")
        df = self.load_data()

        # 2. Create GRU Features
        self.logger.info("\n[STEP 2] Creating GRU-Specific Features")
        df = self.create_gru_features(df)

        # 3. Train/Val/Test Split
        self.logger.info("\n[STEP 3] Splitting Data (70/15/15)")
        (
            X_train,
            y_train,
            X_val,
            y_val,
            X_test,
            y_test,
        ) = self.timeseries_train_test_split(
            df,
            train_ratio=0.7,
            val_ratio=0.15,
            test_ratio=0.15,
            sequence_length=sequence_length,
        )

        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test

        # 4. Hyperparameter Tuning
        self.logger.info("\n[STEP 4] Hyperparameter Tuning")
        best_params = self.hyperparameter_tuning(X_train, y_train, X_val, y_val)

        # 5. Train Final Model
        self.logger.info("\n[STEP 5] Training Final GRU Model")
        model = self.train_final_model(X_train, y_train, X_val, y_val, best_params)

        # 6. Evaluation
        self.logger.info("\n[STEP 6] Model Evaluation")

        y_train_pred = model.predict(X_train, verbose=0).flatten()
        y_val_pred = model.predict(X_val, verbose=0).flatten()
        y_test_pred = model.predict(X_test, verbose=0).flatten()

        train_metrics = self.calculate_metrics(y_train, y_train_pred, "Training Set")
        val_metrics = self.calculate_metrics(y_val, y_val_pred, "Validation Set")
        test_metrics = self.calculate_metrics(y_test, y_test_pred, "Test Set")

        # 7. Cross-Validation
        cv_metrics = None
        if perform_cv:
            self.logger.info("\n[STEP 7] Time-Series Cross-Validation")
            cv_metrics = self.time_series_cross_validation(
                df, n_splits=5, sequence_length=sequence_length
            )

        # 8. Save Results
        self.logger.info("\n[STEP 8] Saving Results")

        all_metrics = {
            "model_type": "GRU",
            "train_metrics": train_metrics,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics,
            "cv_metrics": cv_metrics if cv_metrics else [],
            "best_hyperparameters": best_params,
            "training_history": {
                "loss": [float(x) for x in self.history["loss"]],
                "val_loss": [float(x) for x in self.history["val_loss"]],
            },
        }

        metrics_path = os.path.join(self.output_dir, "metrics", "all_metrics.json")
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(all_metrics, f, indent=4)

        self.logger.info(f"All metrics saved to {metrics_path}")

        predictions_df = pd.DataFrame(
            {
                "y_true": self.scaler_y.inverse_transform(
                    y_test.reshape(-1, 1)
                ).flatten(),
                "y_pred": self.scaler_y.inverse_transform(
                    y_test_pred.reshape(-1, 1)
                ).flatten(),
            }
        )
        pred_path = os.path.join(self.output_dir, "test_predictions.csv")
        predictions_df.to_csv(pred_path, index=False)
        self.logger.info(f"Test predictions saved to {pred_path}")

        self.logger.info("\n" + "=" * 80)
        self.logger.info("GRU MODEL EXPLORATION PIPELINE COMPLETED SUCCESSFULLY")
        self.logger.info("=" * 80)

        return all_metrics


def main():
    """Main execution function."""
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = "gru_output"
    SEQUENCE_LENGTH = 30
    PERFORM_CV = True

    print("=" * 80)
    print("GRU Model Explorer for Sri Lankan Tourism Arrivals")
    print("=" * 80)
    print("\nGRU is a lighter alternative to LSTM with fewer parameters")
    print("and often comparable performance.")
    print("\nConfiguration:")
    print(f"  Data Path       : {DATA_PATH}")
    print(f"  Output Directory: {OUTPUT_DIR}")
    print(f"  Sequence Length : {SEQUENCE_LENGTH} days")
    print(f"  Cross-Validation: {PERFORM_CV}")
    print("=" * 80)

    explorer = GRUModelExplorer(data_path=DATA_PATH, output_dir=OUTPUT_DIR)

    metrics = explorer.run_full_pipeline(
        sequence_length=SEQUENCE_LENGTH, perform_cv=PERFORM_CV
    )

    print("\n" + "=" * 80)
    print("EXECUTION COMPLETED")
    print("=" * 80)
    print("\nFinal Test Set Metrics:")
    for metric, value in metrics["test_metrics"].items():
        print(f"  {metric}: {value:.4f}")

    print(f"\nOutputs saved to: {OUTPUT_DIR}/")
    print("  - models/best_gru_model.h5")
    print("  - metrics/all_metrics.json")
    print("  - best_hyperparameters.json")
    print("  - test_predictions.csv")


if __name__ == "__main__":
    main()

2026-03-18 05:11:50,771 - GRUModelExplorer - INFO - Output directory created: gru_output
INFO:GRUModelExplorer:Output directory created: gru_output
2026-03-18 05:11:50,772 - GRUModelExplorer - INFO - GRU Model Explorer initialized successfully
INFO:GRUModelExplorer:GRU Model Explorer initialized successfully
2026-03-18 05:11:50,773 - GRUModelExplorer - INFO - ================================================================================
INFO:GRUModelExplorer:================================================================================
2026-03-18 05:11:50,774 - GRUModelExplorer - INFO - STARTING GRU MODEL EXPLORATION PIPELINE
INFO:GRUModelExplorer:STARTING GRU MODEL EXPLORATION PIPELINE
2026-03-18 05:11:50,777 - GRUModelExplorer - INFO - ================================================================================
INFO:GRUModelExplorer:================================================================================
2026-03-18 05:11:50,778 - GRUModelExplorer - INFO - 
[STEP 1] Lo

GRU Model Explorer for Sri Lankan Tourism Arrivals

GRU is a lighter alternative to LSTM with fewer parameters
and often comparable performance.

Configuration:
  Data Path       : preprocessed-dataset.csv
  Output Directory: gru_output
  Sequence Length : 30 days
  Cross-Validation: True


2026-03-18 05:12:21,377 - GRUModelExplorer - INFO - Validation loss: 0.007402
INFO:GRUModelExplorer:Validation loss: 0.007402
2026-03-18 05:12:21,380 - GRUModelExplorer - INFO - *** New best parameters found! Val loss: 0.007402 ***
INFO:GRUModelExplorer:*** New best parameters found! Val loss: 0.007402 ***
2026-03-18 05:12:21,382 - GRUModelExplorer - INFO - 
Trial 2/24
INFO:GRUModelExplorer:
Trial 2/24
2026-03-18 05:12:21,384 - GRUModelExplorer - INFO - Params: gru_units=[128, 64], dropout=0.2, lr=0.001, batch_size=64
INFO:GRUModelExplorer:Params: gru_units=[128, 64], dropout=0.2, lr=0.001, batch_size=64
2026-03-18 05:12:37,645 - GRUModelExplorer - INFO - Validation loss: 0.006852
INFO:GRUModelExplorer:Validation loss: 0.006852
2026-03-18 05:12:37,648 - GRUModelExplorer - INFO - *** New best parameters found! Val loss: 0.006852 ***
INFO:GRUModelExplorer:*** New best parameters found! Val loss: 0.006852 ***
2026-03-18 05:12:37,651 - GRUModelExplorer - INFO - 
Trial 3/24
INFO:GRUModelExp

2026-03-18 05:27:29,027 - GRUModelExplorer - INFO - Model: "GRU_Tourism_Predictor"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_1 (GRU)                     │ (None, 30, 256)        │       226,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 30, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 30, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 128)            │       148,224 │
├─────────────────────────────────┼────────────────────────┼──────────

Epoch 1/200
122/125 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 1.7283 - mae: 1.0313
Epoch 1: val_loss improved from None to 0.09145, saving model to gru_output/models/best_gru_model.h5


...


Epoch 44: finished saving model to gru_output/models/best_gru_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0060 - mae: 0.0583 - val_loss: 0.0113 - val_mae: 0.0758 - learning_rate: 1.2500e-04
Epoch 45/200
120/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0060 - mae: 0.0587
Epoch 45: val_loss improved from 0.01128 to 0.01101, saving model to gru_output/models/best_gru_model.h5



Epoch 45: finished saving model to gru_output/models/best_gru_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0058 - mae: 0.0575 - val_loss: 0.0110 - val_mae: 0.0775 - learning_rate: 1.2500e-04
Epoch 46/200
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0057 - mae: 0.0569
Epoch 46: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0059 - mae: 0.0575 - val_loss: 0.0117 - val_mae: 0.0812 - learning_rate: 1.2500e-04
Epoch 47/200
120/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0054 - mae: 0.0560
Epoch 47: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0055 - mae: 0.0562 - val_loss: 0.0119 - val_mae: 0.0812 - learning_rate: 1.2500e-04
Epoch 48/200
120/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0056 - mae: 0.0563
Epoch 48: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0056 - mae: 0.0559 - val_loss: 0.0113 - val_mae: 0.0768 - learning_rate: 1.


Epoch 59: finished saving model to gru_output/models/best_gru_model.h5

Epoch 59: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0048 - mae: 0.0517 - val_loss: 0.0110 - val_mae: 0.0765 - learning_rate: 6.2500e-05
Epoch 60/200
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0052 - mae: 0.0546
Epoch 60: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0053 - mae: 0.0544 - val_loss: 0.0115 - val_mae: 0.0776 - learning_rate: 3.1250e-05
Epoch 61/200
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0048 - mae: 0.0520
Epoch 61: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0049 - mae: 0.0520 - val_loss: 0.0117 - val_mae: 0.0783 - learning_rate: 3.1250e-05
Epoch 62/200
124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0048 - mae: 0.0524
Epoch 62: val_loss did not improve from 0.01101
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss:

2026-03-18 05:29:34,174 - GRUModelExplorer - INFO - Training completed
INFO:GRUModelExplorer:Training completed
2026-03-18 05:29:34,176 - GRUModelExplorer - INFO - 
[STEP 6] Model Evaluation
INFO:GRUModelExplorer:
[STEP 6] Model Evaluation
2026-03-18 05:29:35,259 - GRUModelExplorer - INFO - 
Training Set Metrics:
INFO:GRUModelExplorer:
Training Set Metrics:
2026-03-18 05:29:35,262 - GRUModelExplorer - INFO -   MSE: 289570.7332
INFO:GRUModelExplorer:  MSE: 289570.7332
2026-03-18 05:29:35,263 - GRUModelExplorer - INFO -   RMSE: 538.1178
INFO:GRUModelExplorer:  RMSE: 538.1178
2026-03-18 05:29:35,265 - GRUModelExplorer - INFO -   R2: 0.9465
INFO:GRUModelExplorer:  R2: 0.9465
2026-03-18 05:29:35,266 - GRUModelExplorer - INFO -   MAPE: 2694300768799011840.0000
INFO:GRUModelExplorer:  MAPE: 2694300768799011840.0000
2026-03-18 05:29:35,270 - GRUModelExplorer - INFO - 
Validation Set Metrics:
INFO:GRUModelExplorer:
Validation Set Metrics:
2026-03-18 05:29:35,271 - GRUModelExplorer - INFO -   MS


EXECUTION COMPLETED

Final Test Set Metrics:
  MSE: 1496376.2113
  RMSE: 1223.2646
  R2: 0.6588
  MAPE: 14.7124

Outputs saved to: gru_output/
  - models/best_gru_model.h5
  - metrics/all_metrics.json
  - best_hyperparameters.json
  - test_predictions.csv


In [ ]:
!pip install pytorch-forecasting
!pip install pytorch-lightning

# Additional useful dependencies
!pip install pandas numpy matplotlib
!pip install tensorboard  # for logging

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.8/399.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 76.2 MB/s eta 0:00:00


In [ ]:
# For TSFormer via PyPOTS
!pip install pypots

# Additional dependencies often needed
!pip install scikit-learn pandas numpy
!pip install torch torchvision torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.2/769.2 kB 20.5 MB/s eta 0:00:00


In [ ]:
"""
Time Series Transformer (TSformer) Model Exploration for Sri Lankan Tourism Arrivals Prediction
================================================================================================
This module implements a production-ready Transformer model with hyperparameter tuning,
time-series aware validation, and comprehensive evaluation metrics.
TSformer uses pure attention mechanisms without recurrence, making it highly parallelizable
and effective at capturing long-range dependencies in time series data.
Author: ML Research Team
Date: December 2025
"""
import numpy as np
import pandas as pd
import logging
import json
import warnings
from datetime import datetime
from typing import Tuple, Dict, List, Any
import os
import random
# Deep Learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, LayerNormalization, MultiHeadAttention,
    Input, Add, GlobalAveragePooling1D, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
# Suppress warnings
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

class TSformerModelExplorer:
    """
    Time Series Transformer Model Explorer for time-series forecasting
    with comprehensive preprocessing, hyperparameter tuning, and evaluation.
    TSformer Architecture:
    - Positional encoding for temporal information
    - Multi-head self-attention for capturing dependencies
    - Feed-forward networks for feature transformation
    - Multiple encoder layers for hierarchical feature learning
    """
    def __init__(self, data_path: str, output_dir: str = "tsformer_output"):
        """
        Initialize the TSformer Model Explorer.
        Args:
            data_path: Path to preprocessed dataset CSV
            output_dir: Directory for saving outputs
        """
        self.data_path = data_path
        self.output_dir = output_dir
        self.setup_logging()
        self.setup_output_directory()
        # Model artifacts
        self.scaler_X = None
        self.scaler_y = None
        self.best_model = None
        self.best_params = None
        self.history = None
        # Data splits
        self.X_train, self.X_val, self.X_test = None, None, None
        self.y_train, self.y_val, self.y_test = None, None, None
        self.train_dates, self.val_dates, self.test_dates = None, None, None
        self.logger.info("TSformer Model Explorer initialized successfully")

    # -------------------------------------------------------------------------
    # Logging & IO
    # -------------------------------------------------------------------------
    def setup_logging(self):
        """Configure logging with both file and console handlers."""
        log_format = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        self.logger = logging.getLogger("TSformerModelExplorer")
        self.logger.setLevel(logging.INFO)
        self.logger.handlers = []
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(logging.Formatter(log_format))
        file_handler = logging.FileHandler(
            f"tsformer_exploration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
        )
        file_handler.setLevel(logging.INFO)
        file_handler.setFormatter(logging.Formatter(log_format))
        self.logger.addHandler(console_handler)
        self.logger.addHandler(file_handler)

    def setup_output_directory(self):
        """Create output directory structure."""
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "models"), exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "metrics"), exist_ok=True)
        self.logger.info(f"Output directory created: {self.output_dir}")

    def load_data(self) -> pd.DataFrame:
        """Load and validate preprocessed dataset."""
        self.logger.info(f"Loading data from {self.data_path}")
        try:
            df = pd.read_csv(self.data_path)
            self.logger.info(f"Data loaded successfully. Shape: {df.shape}")
            self.logger.info(f"Columns: {list(df.columns)}")
            self.logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            return df
        except Exception as e:
            self.logger.error(f"Error loading data: {str(e)}")
            raise

    # -------------------------------------------------------------------------
    # Feature Engineering & Sequences
    # -------------------------------------------------------------------------
    def create_tsformer_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Create TSformer-specific features including temporal and contextual features.
        Args:
            df: Input dataframe
        Returns:
            DataFrame with TSformer-specific features
        """
        self.logger.info("Creating TSformer-specific features")
        df = df.copy()
        # Temporal features
        df["day_of_week"] = df["date"].dt.dayofweek
        df["day_of_month"] = df["date"].dt.day
        df["month"] = df["date"].dt.month
        df["quarter"] = df["date"].dt.quarter
        df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
        df["year"] = df["date"].dt.year
        df["day_of_year"] = df["date"].dt.dayofyear
        # Cyclical encoding (important for transformers)
        df["day_of_week_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
        df["day_of_week_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["day_of_year_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
        df["day_of_year_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)
        # Lag features
        for lag in [1, 7, 14, 30]:
            df[f"arrivals_lag_{lag}"] = df["arrivals"].shift(lag)
        # Rolling statistics
        for window in [7, 14, 30]:
            df[f"arrivals_rolling_mean_{window}"] = df["arrivals"].rolling(
                window=window, min_periods=1
            ).mean()
            df[f"arrivals_rolling_std_{window}"] = df["arrivals"].rolling(
                window=window, min_periods=1
            ).std()
        # Exponential moving average
        df["arrivals_ema_7"] = df["arrivals"].ewm(span=7, adjust=False).mean()
        df["arrivals_ema_30"] = df["arrivals"].ewm(span=30, adjust=False).mean()
        # Trend features
        df["arrivals_diff_1"] = df["arrivals"].diff(1)
        df["arrivals_diff_7"] = df["arrivals"].diff(7)
        # Fill NaNs
        df = df.fillna(method="ffill").fillna(method="bfill")
        self.logger.info(f"TSformer features created. New shape: {df.shape}")
        return df

    def prepare_sequences(self, data: np.ndarray, sequence_length: int) -> np.ndarray:
        """
        Prepare sequences for TSformer input.
        Args:
            data: Input data array (n_samples, n_features)
            sequence_length: Length of sequences (lookback window)
        Returns:
            Array of sequences (n_sequences, sequence_length, n_features)
        """
        sequences = []
        for i in range(len(data) - sequence_length + 1):
            sequences.append(data[i : i + sequence_length])
        return np.array(sequences)

    # -------------------------------------------------------------------------
    # Train/Val/Test Split
    # -------------------------------------------------------------------------
    def timeseries_train_test_split(
        self,
        df: pd.DataFrame,
        train_ratio: float = 0.7,
        val_ratio: float = 0.15,
        test_ratio: float = 0.15,
        sequence_length: int = 30,
    ) -> Tuple:
        """
        Perform time-series aware train/validation/test split with sequence creation.
        Args:
            df: Input dataframe
            train_ratio: Proportion for training set
            val_ratio: Proportion for validation set
            test_ratio: Proportion for test set
            sequence_length: Lookback window for TSformer
        Returns:
            Tuple of train, validation, and test sets
        """
        self.logger.info(
            f"Performing time-series split: {train_ratio}/{val_ratio}/{test_ratio}"
        )
        self.logger.info(f"Sequence length (lookback): {sequence_length}")
        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"
        n = len(df)
        train_end = int(n * train_ratio)
        val_end = int(n * (train_ratio + val_ratio))
        train_df = df.iloc[:train_end].copy()
        val_df = df.iloc[train_end:val_end].copy()
        test_df = df.iloc[val_end:].copy()
        self.logger.info(
            f"Train set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})"
        )
        self.logger.info(
            f"Validation set: {len(val_df)} samples ({val_df['date'].min()} to {val_df['date'].max()})"
        )
        self.logger.info(
            f"Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})"
        )
        self.train_dates = train_df["date"].values
        self.val_dates = val_df["date"].values
        self.test_dates = test_df["date"].values
        feature_cols = [
            col
            for col in df.columns
            if col not in ["date", "arrivals", "arrivals_robust_scaled", "outlier_flag"]
        ]
        X_train = train_df[feature_cols].values
        X_val = val_df[feature_cols].values
        X_test = test_df[feature_cols].values
        y_train = train_df["arrivals"].values
        y_val = val_df["arrivals"].values
        y_test = test_df["arrivals"].values
        # Scaling
        self.logger.info("Scaling features using StandardScaler")
        self.scaler_X = StandardScaler()
        X_train_scaled = self.scaler_X.fit_transform(X_train)
        X_val_scaled = self.scaler_X.transform(X_val)
        X_test_scaled = self.scaler_X.transform(X_test)
        self.logger.info("Scaling target using MinMaxScaler")
        self.scaler_y = MinMaxScaler()
        y_train_scaled = self.scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val_scaled = self.scaler_y.transform(y_val.reshape(-1, 1)).flatten()
        y_test_scaled = self.scaler_y.transform(y_test.reshape(-1, 1)).flatten()
        # Sequences
        self.logger.info("Creating sequences for TSformer")
        X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
        X_val_seq = self.prepare_sequences(X_val_scaled, sequence_length)
        X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)
        y_train_seq = y_train_scaled[sequence_length - 1 :]
        y_val_seq = y_val_scaled[sequence_length - 1 :]
        y_test_seq = y_test_scaled[sequence_length - 1 :]
        self.logger.info(
            f"Final shapes - X_train: {X_train_seq.shape}, y_train: {y_train_seq.shape}"
        )
        self.logger.info(
            f"Final shapes - X_val: {X_val_seq.shape}, y_val: {y_val_seq.shape}"
        )
        self.logger.info(
            f"Final shapes - X_test: {X_test_seq.shape}, y_test: {y_test_seq.shape}"
        )
        return X_train_seq, y_train_seq, X_val_seq, y_val_seq, X_test_seq, y_test_seq

    # -------------------------------------------------------------------------
    # Transformer Architecture Components
    # -------------------------------------------------------------------------
    def positional_encoding(self, sequence_length: int, d_model: int):
        """
        Generate positional encoding for transformer.
        Args:
            sequence_length: Length of input sequence
            d_model: Dimension of model (embedding size)
        Returns:
            Positional encoding tensor
        """
        positions = np.arange(sequence_length)[:, np.newaxis]
        dimensions = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates
        # Apply sin to even indices, cos to odd indices
        sines = np.sin(angle_rads[:, 0::2])
        cosines = np.cos(angle_rads[:, 1::2])
        pos_encoding = np.zeros((sequence_length, d_model))
        pos_encoding[:, 0::2] = sines
        pos_encoding[:, 1::2] = cosines
        return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

    def transformer_encoder_block(
        self,
        inputs,
        d_model: int,
        num_heads: int,
        ff_dim: int,
        dropout_rate: float,
        name: str,
    ):
        """
        Single Transformer encoder block.
        Args:
            inputs: Input tensor
            d_model: Model dimension
            num_heads: Number of attention heads
            ff_dim: Feed-forward dimension
            dropout_rate: Dropout rate
            name: Block name prefix
        Returns:
            Output tensor
        """
        # Multi-head self-attention
        attention_output = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout_rate,
            name=f"{name}_attention"
        )(inputs, inputs)
        attention_output = Dropout(dropout_rate, name=f"{name}_attn_dropout")(attention_output)
        # Add & Norm
        out1 = Add(name=f"{name}_add1")([inputs, attention_output])
        out1 = LayerNormalization(epsilon=1e-6, name=f"{name}_norm1")(out1)
        # Feed-forward network
        ffn_output = Dense(ff_dim, activation="relu", name=f"{name}_ffn1")(out1)
        ffn_output = Dropout(dropout_rate, name=f"{name}_ffn_dropout1")(ffn_output)
        ffn_output = Dense(d_model, name=f"{name}_ffn2")(ffn_output)
        ffn_output = Dropout(dropout_rate, name=f"{name}_ffn_dropout2")(ffn_output)
        # Add & Norm
        out2 = Add(name=f"{name}_add2")([out1, ffn_output])
        out2 = LayerNormalization(epsilon=1e-6, name=f"{name}_norm2")(out2)
        return out2

    def build_tsformer_model(
        self,
        input_shape: Tuple[int, int],
        d_model: int = 128,
        num_heads: int = 8,
        num_encoder_layers: int = 4,
        ff_dim: int = 256,
        dropout_rate: float = 0.1,
        learning_rate: float = 0.0001,
    ) -> keras.Model:
        """
        Build Time Series Transformer model architecture.
        Args:
            input_shape: Shape of input (sequence_length, n_features)
            d_model: Model dimension (embedding size)
            num_heads: Number of attention heads
            num_encoder_layers: Number of encoder blocks
            ff_dim: Feed-forward network dimension
            dropout_rate: Dropout rate for regularization
            learning_rate: Learning rate for optimizer
        Returns:
            Compiled Keras model
        """
        sequence_length, num_features = input_shape
        # Input
        inputs = Input(shape=input_shape, name="input")
        # Project input features to d_model dimension
        x = Dense(d_model, name="input_projection")(inputs)
        # Add positional encoding
        pos_encoding = self.positional_encoding(sequence_length, d_model)
        x = x + pos_encoding
        x = Dropout(dropout_rate, name="pos_dropout")(x)
        # Stack encoder blocks
        for i in range(num_encoder_layers):
            x = self.transformer_encoder_block(
                x,
                d_model=d_model,
                num_heads=num_heads,
                ff_dim=ff_dim,
                dropout_rate=dropout_rate,
                name=f"encoder_{i+1}",
            )
        # Global pooling
        x = GlobalAveragePooling1D(name="global_avg_pool")(x)
        # Dense layers for prediction
        x = Dense(64, activation="relu", name="dense_1")(x)
        x = Dropout(dropout_rate, name="dense_dropout_1")(x)
        # Output
        outputs = Dense(1, activation="linear", name="output")(x)
        # Build model
        model = Model(inputs=inputs, outputs=outputs, name="TSformer_Tourism_Predictor")
        # Compile
        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
        return model

    # -------------------------------------------------------------------------
    # Hyperparameter Tuning & Training
    # -------------------------------------------------------------------------
    def hyperparameter_tuning(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
    ) -> Dict[str, Any]:
        """
        Simple random-search hyperparameter tuning for TSformer (5 trials).
        Limits epochs and patience for speed while still finding good params.
        """
        self.logger.info("Starting hyperparameter tuning (random search - 5 trials)")
        param_grid = {
            "d_model": [64, 128],
            "num_heads": [4, 8],
            "num_encoder_layers": [2, 4],
            "ff_dim": [128, 256],
            "dropout_rate": [0.1, 0.2],
            "learning_rate": [0.0001, 0.001],
            "batch_size": [32, 64],
        }
        n_trials = 5
        best_val_loss = float("inf")
        best_params = None

        for trial in range(n_trials):
            params = {k: random.choice(v) for k, v in param_grid.items()}
            self.logger.info(f"Trial {trial+1}/{n_trials}: {params}")

            model = self.build_tsformer_model(
                input_shape=(X_train.shape[1], X_train.shape[2]),
                d_model=params["d_model"],
                num_heads=params["num_heads"],
                num_encoder_layers=params["num_encoder_layers"],
                ff_dim=params["ff_dim"],
                dropout_rate=params["dropout_rate"],
                learning_rate=params["learning_rate"],
            )

            early_stop = EarlyStopping(
                monitor="val_loss", patience=5, restore_best_weights=True, verbose=0
            )
            reduce_lr = ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=3, verbose=0
            )

            history = model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                epochs=20,  # short tuning epochs
                batch_size=params["batch_size"],
                callbacks=[early_stop, reduce_lr],
                verbose=0,
            )

            val_loss = min(history.history["val_loss"])
            self.logger.info(f"Trial {trial+1} val_loss: {val_loss:.6f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = params.copy()

        self.best_params = best_params
        self.logger.info(f"Best hyperparameters found: {best_params} (val_loss={best_val_loss:.6f})")

        # Save best params
        params_path = os.path.join(self.output_dir, "best_hyperparameters.json")
        with open(params_path, "w", encoding="utf-8") as f:
            json.dump(best_params, f, indent=4)
        self.logger.info(f"Best hyperparameters saved to {params_path}")

        return best_params

    def train_final_model(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_val: np.ndarray,
        y_val: np.ndarray,
        best_params: Dict[str, Any],
    ) -> keras.Model:
        """
        Train the final TSformer model using the best hyperparameters.
        Includes full training (up to 100 epochs) with early stopping, LR reduction,
        and model checkpointing.
        """
        self.logger.info("Training final TSformer model with best hyperparameters")

        model = self.build_tsformer_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            d_model=best_params["d_model"],
            num_heads=best_params["num_heads"],
            num_encoder_layers=best_params["num_encoder_layers"],
            ff_dim=best_params["ff_dim"],
            dropout_rate=best_params["dropout_rate"],
            learning_rate=best_params["learning_rate"],
        )

        early_stop = EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True, verbose=1
        )
        reduce_lr = ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
        )
        checkpoint = ModelCheckpoint(
            os.path.join(self.output_dir, "models", "best_tsformer_model.h5"),
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        )

        history_obj = model.fit(
            X_train,
            y_train,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=best_params.get("batch_size", 32),
            callbacks=[early_stop, reduce_lr, checkpoint],
            verbose=1,
        )

        self.history = history_obj.history
        self.best_model = model
        self.logger.info("Final model training completed and best weights restored")
        return model

    # -------------------------------------------------------------------------
    # Evaluation & CV
    # -------------------------------------------------------------------------
    def calculate_metrics(
        self, y_true: np.ndarray, y_pred: np.ndarray, set_name: str = ""
    ) -> Dict[str, float]:
        """Calculate evaluation metrics on original scale."""
        y_true_original = self.scaler_y.inverse_transform(y_true.reshape(-1, 1)).flatten()
        y_pred_original = self.scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()
        mse = mean_squared_error(y_true_original, y_pred_original)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_true_original, y_pred_original)
        mape = mean_absolute_percentage_error(y_true_original, y_pred_original) * 100
        metrics = {
            "MSE": float(mse),
            "RMSE": float(rmse),
            "R2": float(r2),
            "MAPE": float(mape),
        }
        self.logger.info(f"\n{set_name} Metrics:")
        for metric, value in metrics.items():
            self.logger.info(f" {metric}: {value:.4f}")
        return metrics

    def time_series_cross_validation(
        self, df: pd.DataFrame, n_splits: int = 3, sequence_length: int = 30
    ) -> List[Dict[str, float]]:
        """Perform time-series cross-validation with TSformer."""
        self.logger.info(f"\nPerforming time-series cross-validation with {n_splits} splits")
        feature_cols = [
            col
            for col in df.columns
            if col not in ["date", "arrivals", "arrivals_robust_scaled", "outlier_flag"]
        ]
        n = len(df)
        fold_size = n // (n_splits + 1)
        cv_metrics = []
        for fold in range(n_splits):
            self.logger.info(f"\n--- Fold {fold + 1}/{n_splits} ---")
            test_start = fold_size * (fold + 1)
            test_end = test_start + fold_size
            train_df = df.iloc[:test_start].copy()
            test_df = df.iloc[test_start:test_end].copy()
            if len(train_df) < sequence_length or len(test_df) < sequence_length:
                self.logger.warning(f"Fold {fold + 1} skipped due to insufficient data")
                continue
            X_train = train_df[feature_cols].values
            X_test = test_df[feature_cols].values
            y_train = train_df["arrivals"].values
            y_test = test_df["arrivals"].values
            scaler_X = StandardScaler()
            scaler_y = MinMaxScaler()
            X_train_scaled = scaler_X.fit_transform(X_train)
            X_test_scaled = scaler_X.transform(X_test)
            y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
            y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()
            X_train_seq = self.prepare_sequences(X_train_scaled, sequence_length)
            X_test_seq = self.prepare_sequences(X_test_scaled, sequence_length)
            y_train_seq = y_train_scaled[sequence_length - 1 :]
            y_test_seq = y_test_scaled[sequence_length - 1 :]
            if self.best_params:
                params = self.best_params
            else:
                params = {
                    "d_model": 128,
                    "num_heads": 8,
                    "num_encoder_layers": 4,
                    "ff_dim": 256,
                    "dropout_rate": 0.1,
                    "learning_rate": 0.0001,
                    "batch_size": 32,
                }
            try:
                model = self.build_tsformer_model(
                    input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]),
                    d_model=params["d_model"],
                    num_heads=params["num_heads"],
                    num_encoder_layers=params["num_encoder_layers"],
                    ff_dim=params["ff_dim"],
                    dropout_rate=params["dropout_rate"],
                    learning_rate=params["learning_rate"],
                )
                early_stop = EarlyStopping(
                    monitor="loss", patience=10, restore_best_weights=True, verbose=0
                )
                model.fit(
                    X_train_seq,
                    y_train_seq,
                    epochs=30,
                    batch_size=params["batch_size"],
                    callbacks=[early_stop],
                    verbose=0,
                )
                y_pred = model.predict(X_test_seq, verbose=0).flatten()
                y_true_original = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
                y_pred_original = scaler_y.inverse_transform(y_pred.reshape(-1, 1)).flatten()
                fold_metrics = {
                    "fold": fold + 1,
                    "MSE": float(mean_squared_error(y_true_original, y_pred_original)),
                    "RMSE": float(
                        np.sqrt(mean_squared_error(y_true_original, y_pred_original))
                    ),
                    "R2": float(r2_score(y_true_original, y_pred_original)),
                    "MAPE": float(
                        mean_absolute_percentage_error(y_true_original, y_pred_original) * 100
                    ),
                }
                cv_metrics.append(fold_metrics)
                self.logger.info(f"Fold {fold + 1} metrics:")
                for metric, value in fold_metrics.items():
                    if metric != "fold":
                        self.logger.info(f" {metric}: {value:.4f}")
                keras.backend.clear_session()
            except Exception as e:
                self.logger.warning(f"Fold {fold + 1} failed: {str(e)}")
                continue
        if cv_metrics:
            avg_metrics = {
                "MSE": float(np.mean([m["MSE"] for m in cv_metrics])),
                "RMSE": float(np.mean([m["RMSE"] for m in cv_metrics])),
                "R2": float(np.mean([m["R2"] for m in cv_metrics])),
                "MAPE": float(np.mean([m["MAPE"] for m in cv_metrics])),
            }
            self.logger.info("\nAverage CV Metrics:")
            for metric, value in avg_metrics.items():
                self.logger.info(f" {metric}: {value:.4f}")
        return cv_metrics

    # -------------------------------------------------------------------------
    # Full Pipeline
    # -------------------------------------------------------------------------
    def run_full_pipeline(self, sequence_length: int = 30, perform_cv: bool = True):
        """Execute the complete TSformer model exploration pipeline."""
        self.logger.info("=" * 80)
        self.logger.info("STARTING TSFORMER MODEL EXPLORATION PIPELINE")
        self.logger.info("=" * 80)
        # 1. Load Data
        self.logger.info("\n[STEP 1] Loading Data")
        df = self.load_data()
        # 2. Create TSformer Features
        self.logger.info("\n[STEP 2] Creating TSformer-Specific Features")
        df = self.create_tsformer_features(df)
        # 3. Train/Val/Test Split
        self.logger.info("\n[STEP 3] Splitting Data (70/15/15)")
        (
            X_train,
            y_train,
            X_val,
            y_val,
            X_test,
            y_test,
        ) = self.timeseries_train_test_split(
            df,
            train_ratio=0.7,
            val_ratio=0.15,
            test_ratio=0.15,
            sequence_length=sequence_length,
        )
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test
        # 4. Hyperparameter Tuning
        self.logger.info("\n[STEP 4] Hyperparameter Tuning")
        best_params = self.hyperparameter_tuning(X_train, y_train, X_val, y_val)
        # 5. Train Final Model
        self.logger.info("\n[STEP 5] Training Final TSformer Model")
        model = self.train_final_model(X_train, y_train, X_val, y_val, best_params)
        # 6. Evaluation
        self.logger.info("\n[STEP 6] Model Evaluation")
        y_train_pred = model.predict(X_train, verbose=0).flatten()
        y_val_pred = model.predict(X_val, verbose=0).flatten()
        y_test_pred = model.predict(X_test, verbose=0).flatten()
        train_metrics = self.calculate_metrics(y_train, y_train_pred, "Training Set")
        val_metrics = self.calculate_metrics(y_val, y_val_pred, "Validation Set")
        test_metrics = self.calculate_metrics(y_test, y_test_pred, "Test Set")
        # 7. Cross-Validation
        cv_metrics = None
        if perform_cv:
            self.logger.info("\n[STEP 7] Time-Series Cross-Validation")
            cv_metrics = self.time_series_cross_validation(
                df, n_splits=3, sequence_length=sequence_length
            )
        # 8. Save Results
        self.logger.info("\n[STEP 8] Saving Results")
        all_metrics = {
            "model_type": "TSformer",
            "train_metrics": train_metrics,
            "validation_metrics": val_metrics,
            "test_metrics": test_metrics,
            "cv_metrics": cv_metrics if cv_metrics else [],
            "best_hyperparameters": best_params,
            "training_history": {
                "loss": [float(x) for x in self.history["loss"]],
                "val_loss": [float(x) for x in self.history["val_loss"]],
            },
        }
        metrics_path = os.path.join(self.output_dir, "metrics", "all_metrics.json")
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(all_metrics, f, indent=4)
        self.logger.info(f"All metrics saved to {metrics_path}")
        predictions_df = pd.DataFrame(
            {
                "y_true": self.scaler_y.inverse_transform(
                    y_test.reshape(-1, 1)
                ).flatten(),
                "y_pred": self.scaler_y.inverse_transform(
                    y_test_pred.reshape(-1, 1)
                ).flatten(),
            }
        )
        pred_path = os.path.join(self.output_dir, "test_predictions.csv")
        predictions_df.to_csv(pred_path, index=False)
        self.logger.info(f"Test predictions saved to {pred_path}")
        self.logger.info("\n" + "=" * 80)
        self.logger.info("TSFORMER MODEL EXPLORATION PIPELINE COMPLETED SUCCESSFULLY")
        self.logger.info("=" * 80)
        return all_metrics


def main():
    """Main execution function."""
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = "tsformer_output"
    SEQUENCE_LENGTH = 30
    PERFORM_CV = True
    print("=" * 80)
    print("Time Series Transformer (TSformer) Model Explorer")
    print("Sri Lankan Tourism Arrivals Prediction")
    print("=" * 80)
    print("\nTSformer uses pure attention mechanisms without recurrence,")
    print("making it highly parallelizable and effective for long-range dependencies.")
    print("\nKey Features:")
    print(" • Positional encoding for temporal information")
    print(" • Multi-head self-attention for capturing dependencies")
    print(" • Multiple encoder layers for hierarchical learning")
    print(" • Feed-forward networks for feature transformation")
    print(" • Highly parallelizable architecture")
    print("\nConfiguration:")
    print(f" Data Path : {DATA_PATH}")
    print(f" Output Directory: {OUTPUT_DIR}")
    print(f" Sequence Length : {SEQUENCE_LENGTH} days")
    print(f" Cross-Validation: {PERFORM_CV}")
    print("=" * 80)
    explorer = TSformerModelExplorer(data_path=DATA_PATH, output_dir=OUTPUT_DIR)
    metrics = explorer.run_full_pipeline(
        sequence_length=SEQUENCE_LENGTH, perform_cv=PERFORM_CV
    )
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETED")
    print("=" * 80)
    print("\nFinal Test Set Metrics:")
    for metric, value in metrics["test_metrics"].items():
        print(f" {metric}: {value:.4f}")
    print(f"\nOutputs saved to: {OUTPUT_DIR}/")
    print(" - models/best_tsformer_model.h5")
    print(" - metrics/all_metrics.json")
    print(" - best_hyperparameters.json")
    print(" - test_predictions.csv")
    print("\nNote: Transformer models benefit from longer training and larger datasets.")


if __name__ == "__main__":
    main()

2026-03-18 05:54:10,530 - TSformerModelExplorer - INFO - Output directory created: tsformer_output
INFO:TSformerModelExplorer:Output directory created: tsformer_output
2026-03-18 05:54:10,532 - TSformerModelExplorer - INFO - TSformer Model Explorer initialized successfully
INFO:TSformerModelExplorer:TSformer Model Explorer initialized successfully
2026-03-18 05:54:10,533 - TSformerModelExplorer - INFO - ================================================================================
INFO:TSformerModelExplorer:================================================================================
2026-03-18 05:54:10,534 - TSformerModelExplorer - INFO - STARTING TSFORMER MODEL EXPLORATION PIPELINE
INFO:TSformerModelExplorer:STARTING TSFORMER MODEL EXPLORATION PIPELINE
2026-03-18 05:54:10,536 - TSformerModelExplorer - INFO - ================================================================================
INFO:TSformerModelExplorer:=================================================================

Time Series Transformer (TSformer) Model Explorer
Sri Lankan Tourism Arrivals Prediction

TSformer uses pure attention mechanisms without recurrence,
making it highly parallelizable and effective for long-range dependencies.

Key Features:
 • Positional encoding for temporal information
 • Multi-head self-attention for capturing dependencies
 • Multiple encoder layers for hierarchical learning
 • Feed-forward networks for feature transformation
 • Highly parallelizable architecture

Configuration:
 Data Path : preprocessed-dataset.csv
 Output Directory: tsformer_output
 Sequence Length : 30 days
 Cross-Validation: True


2026-03-18 05:54:10,748 - TSformerModelExplorer - INFO - Final shapes - X_train: (3988, 30, 44), y_train: (3988,)
INFO:TSformerModelExplorer:Final shapes - X_train: (3988, 30, 44), y_train: (3988,)
2026-03-18 05:54:10,751 - TSformerModelExplorer - INFO - Final shapes - X_val: (833, 30, 44), y_val: (833,)
INFO:TSformerModelExplorer:Final shapes - X_val: (833, 30, 44), y_val: (833,)
2026-03-18 05:54:10,752 - TSformerModelExplorer - INFO - Final shapes - X_test: (832, 30, 44), y_test: (832,)
INFO:TSformerModelExplorer:Final shapes - X_test: (832, 30, 44), y_test: (832,)
2026-03-18 05:54:10,754 - TSformerModelExplorer - INFO - 
[STEP 4] Hyperparameter Tuning
INFO:TSformerModelExplorer:
[STEP 4] Hyperparameter Tuning
2026-03-18 05:54:10,755 - TSformerModelExplorer - INFO - Starting hyperparameter tuning (random search - 5 trials)
INFO:TSformerModelExplorer:Starting hyperparameter tuning (random search - 5 trials)
2026-03-18 05:54:10,757 - TSformerModelExplorer - INFO - Trial 1/5: {'d_model'

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 0.3002 - mae: 0.3155
Epoch 1: val_loss improved from None to 0.06129, saving model to tsformer_output/models/best_tsformer_model.h5



Epoch 1: finished saving model to tsformer_output/models/best_tsformer_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 87ms/step - loss: 0.0874 - mae: 0.1563 - val_loss: 0.0613 - val_mae: 0.1788 - learning_rate: 0.0010
Epoch 2/100
117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0086 - mae: 0.0692
Epoch 2: val_loss improved from 0.06129 to 0.02986, saving model to tsformer_output/models/best_tsformer_model.h5


...


Epoch 33: finished saving model to tsformer_output/models/best_tsformer_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0011 - mae: 0.0227 - val_loss: 0.0028 - val_mae: 0.0429 - learning_rate: 0.0010
Epoch 34/100
120/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0011 - mae: 0.0230
Epoch 34: val_loss did not improve from 0.00281
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0011 - mae: 0.0233 - val_loss: 0.0030 - val_mae: 0.0409 - learning_rate: 0.0010
Epoch 35/100
117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0010 - mae: 0.0227
Epoch 35: val_loss did not improve from 0.00281
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0010 - mae: 0.0227 - val_loss: 0.0034 - val_mae: 0.0422 - learning_rate: 0.0010
Epoch 36/100
117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.6491e-04 - mae: 0.0217
Epoch 36: val_loss did not improve from 0.00281
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0010 - mae: 0.0221 - val_loss: 0.0036 - val_mae: 0.0491 - learning_rate: 0.00


Epoch 39: finished saving model to tsformer_output/models/best_tsformer_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 8.9251e-04 - mae: 0.0207 - val_loss: 0.0025 - val_mae: 0.0372 - learning_rate: 5.0000e-04
Epoch 40/100
117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.5866e-04 - mae: 0.0197
Epoch 40: val_loss improved from 0.00254 to 0.00230, saving model to tsformer_output/models/best_tsformer_model.h5



Epoch 40: finished saving model to tsformer_output/models/best_tsformer_model.h5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 9.0005e-04 - mae: 0.0201 - val_loss: 0.0023 - val_mae: 0.0355 - learning_rate: 5.0000e-04
Epoch 41/100
116/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 8.7686e-04 - mae: 0.0200
Epoch 41: val_loss did not improve from 0.00230
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 9.2024e-04 - mae: 0.0205 - val_loss: 0.0028 - val_mae: 0.0390 - learning_rate: 5.0000e-04
Epoch 42/100
123/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.1371e-04 - mae: 0.0210
Epoch 42: val_loss did not improve from 0.00230
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 9.3611e-04 - mae: 0.0211 - val_loss: 0.0027 - val_mae: 0.0381 - learning_rate: 5.0000e-04
Epoch 43/100
117/125 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 9.3715e-04 - mae: 0.0204
Epoch 43: val_loss did not improve from 0.00230
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 8.6874e-04 - mae: 0.0201 - val_loss: 0.0025 - v

2026-03-18 05:59:00,239 - TSformerModelExplorer - INFO - Final model training completed and best weights restored
INFO:TSformerModelExplorer:Final model training completed and best weights restored
2026-03-18 05:59:00,243 - TSformerModelExplorer - INFO - 
[STEP 6] Model Evaluation
INFO:TSformerModelExplorer:
[STEP 6] Model Evaluation
2026-03-18 05:59:06,320 - TSformerModelExplorer - INFO - 
Training Set Metrics:
INFO:TSformerModelExplorer:
Training Set Metrics:
2026-03-18 05:59:06,322 - TSformerModelExplorer - INFO -  MSE: 15690.1790
INFO:TSformerModelExplorer: MSE: 15690.1790
2026-03-18 05:59:06,325 - TSformerModelExplorer - INFO -  RMSE: 125.2604
INFO:TSformerModelExplorer: RMSE: 125.2604
2026-03-18 05:59:06,328 - TSformerModelExplorer - INFO -  R2: 0.9971
INFO:TSformerModelExplorer: R2: 0.9971
2026-03-18 05:59:06,333 - TSformerModelExplorer - INFO -  MAPE: 1422355525349958656.0000
INFO:TSformerModelExplorer: MAPE: 1422355525349958656.0000
2026-03-18 05:59:06,340 - TSformerModelExplo


EXECUTION COMPLETED

Final Test Set Metrics:
 MSE: 666785.5641
 RMSE: 816.5694
 R2: 0.8480
 MAPE: 11.1440

Outputs saved to: tsformer_output/
 - models/best_tsformer_model.h5
 - metrics/all_metrics.json
 - best_hyperparameters.json
 - test_predictions.csv

Note: Transformer models benefit from longer training and larger datasets.
